# Quadriga — Rice Leaf Blast Detection (VGG16) Notebook

It is organised into the modules in the outline: **Pre-processing**, **EDA**, **Dataset Splitting**, **Training (VGG16 baseline)**, **Feature Extraction for PLSR/XGBoost**, **Validation**, **Testing**, and **Reporting**.

## Verify GPU Usage

In [ ]:
import tensorflow as tf

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU detected: {gpus}")
    # Test GPU operation
    with tf.device('/GPU:0'):
        a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
        b = tf.constant([[1.0, 1.0], [0.0, 1.0]])
        c = tf.matmul(a, b)
        print("GPU matrix multiplication test passed!")
        print(c.numpy())
else:
    print("No GPU detected")

## Library and Initializaations

In [ ]:
# Imports and Configuration
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageFilter
import hashlib
import shutil
import time
import json
import joblib
import cv2
import shap

# TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, load_model

# Import KerasTuner
import keras_tuner as kt

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# XGBoost (if available)
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed. PLSR will be used as alternative.")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set paths to datasets
DATA_DIR_FOREIGN = "E:/Codes/Jupytr/datasets/RiceLeaf/GlobalRiceLeaf/shayanriyaz"
DATA_DIR_LOCAL = "E:/Codes/Jupytr/datasets/RiceLeaf/LocalRiceLeaf/ZAMBALI_RICE_DATASET_V3"
OUTPUT_BASE = "E:/Codes/Jupytr/output/Output_Base"

# Create output directory
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Configuration
IMG_EXTS = ('.jpg', '.jpeg', '.png')
TARGET_SIZE = (224, 224)  # VGG16 input size
BATCH_SIZE = 16
EPOCHS = 50

print("Imports and configuration completed successfully.")

### Without Augmentation Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - WITHOUT DATA AUGMENTATION
# =============================================================================

# Set output directory for non-augmented results
OUTPUT_BASE = "E:/Codes/Jupytr/output/Output_Base/without_augmentation"

# Set augmentation flag to False
run_aug = False

# Create output directory
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("="*60)
print("CONFIGURATION: WITHOUT DATA AUGMENTATION")
print("="*60)
print(f"Output directory: {OUTPUT_BASE}")
print(f"Data augmentation enabled: {run_aug}")
print("All results will be saved without augmented data")
print("="*60)

### With Augmentation Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - WITH DATA AUGMENTATION
# =============================================================================

# Set output directory for augmented results
OUTPUT_BASE = "E:/Codes/Jupytr/output/Output_Base/with_augmentation"

# Set augmentation flag to True
run_aug = True

# Create output directory
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("="*60)
print("CONFIGURATION: WITH DATA AUGMENTATION")
print("="*60)
print(f"Output directory: {OUTPUT_BASE}")
print(f"Data augmentation enabled: {run_aug}")
print("Data augmentation will be applied to training set")
print("="*60)

## 1) Pre-processing Module

Steps:

1. Extract RGB images (PNG/JPG) into numpy arrays.
2. Remove duplicates and blurred images (automatic).
3. Annotate/Labeling of Images
4. Generate CSV or JSON with metadata and labels.
5. Perform EDA
6. Data Augmentation for Class Balance

### 1.1 Image Extraction and Processing

In [ ]:
def load_and_preprocess_image(image_path, target_size=TARGET_SIZE):
    """Load and preprocess image for CNN"""
    try:
        image = Image.open(image_path)
        image = image.resize(target_size)
        image_array = np.array(image)

        # Ensure 3 channels
        if len(image_array.shape) == 2:  # Grayscale
            image_array = np.stack([image_array] * 3, axis=-1)
        elif image_array.shape[2] == 4:  # RGBA
            image_array = image_array[:, :, :3]

        return image_array.astype(np.float32) / 255.0  # Normalize to [0,1]
    except Exception as e:
        print(f"Error loading {image_path}: {e}")
        return None

def extract_all_images(data_dir):
    """Extract all images from directory"""
    image_files = []

    for ext in IMG_EXTS:
        # Only search with lowercase, Windows is case-insensitive anyway
        files = Path(data_dir).rglob(f"*{ext}")
        image_files.extend(files)

    # Convert to string and remove any potential duplicates
    unique_files = list(set(str(img_path) for img_path in image_files))

    print(f"Found {len(unique_files)} images in {data_dir}")
    return unique_files

# Extract images from both datasets with the function
foreign_images = extract_all_images(DATA_DIR_FOREIGN)
ph_images = extract_all_images(DATA_DIR_LOCAL)

print("Image extraction completed.")

### 1.2 Duplicate and Blur Detection

In [ ]:
def calculate_image_hash(image_path):
    """Calculate MD5 hash of an image file to identify duplicates"""
    try:
        with open(image_path, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except Exception as e:
        print(f"Error reading {image_path}: {e}")
        return None

def detect_blur_image_pil(image_path, threshold=100):
    """Detect blur using PIL - variance of Laplacian approximation"""
    try:
        image = Image.open(image_path).convert('L')  # Convert to grayscale
        image_array = np.array(image)

        # Calculate Laplacian variance (better blur detection)
        laplacian_var = np.var(image_array)

        # Debug: print some values to see the range
        if np.random.random() < 0.01:  # Print 1% of images for debugging
            print(f"Debug - {Path(image_path).name}: Laplacian variance = {laplacian_var:.2f}")

        return laplacian_var < threshold
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return True

def filter_images(image_paths, dataset_name):
    """Filter images for duplicates and blur"""
    print(f"Filtering {dataset_name} dataset...")

    image_hashes = {}
    valid_images = []
    invalid_images = []

    for img_path in image_paths:
        # Calculate hash for duplicate detection
        img_hash = calculate_image_hash(img_path)
        if img_hash is None:
            invalid_images.append(img_path)
            continue

        # Check for duplicates
        if img_hash in image_hashes:
            invalid_images.append(img_path)
            continue

        # Check for blur
        if detect_blur_image_pil(img_path):
            invalid_images.append(img_path)
            continue

        image_hashes[img_hash] = img_path
        valid_images.append(img_path)

    print(f"Valid images: {len(valid_images)}, Invalid images: {len(invalid_images)}")
    return valid_images, invalid_images

# Filter both datasets
foreign_valid, foreign_invalid = filter_images(foreign_images, "foreign")
ph_valid, ph_invalid = filter_images(ph_images, "Philippines")

print("Duplicate and blur detection completed.")

### 1.3 Image Segmentation

#### Background Removal

#### Grayscale -> CLAHE

In [ ]:
def apply_clahe_grayscale(image_array):
    """Apply CLAHE and convert to grayscale for enhanced feature extraction"""
    try:
        # Convert to 8-bit for OpenCV processing
        img_uint8 = (image_array * 255).astype(np.uint8)
        
        # Convert to grayscale
        gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
        
        # Apply CLAHE for contrast enhancement
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        clahe_gray = clahe.apply(gray)
        
        # Convert back to 3-channel for model compatibility (VGG16 expects 3 channels)
        clahe_rgb = cv2.cvtColor(clahe_gray, cv2.COLOR_GRAY2RGB)
        
        # Convert back to float and normalize
        result_image = clahe_rgb.astype(np.float32) / 255.0
        
        return result_image, clahe_gray
        
    except Exception as e:
        print(f"Error in CLAHE grayscale: {e}")
        return image_array, None

def apply_clahe_to_dataset(valid_images, dataset_name):
    """Apply CLAHE enhancement to all valid images"""
    print(f"Applying CLAHE enhancement to {dataset_name} dataset...")
    
    processed_images = []
    
    for i, img_path in enumerate(valid_images):
        if i % 500 == 0:
            print(f"  Processed {i}/{len(valid_images)} images...")
            
        try:
            # Load original image
            original_img = load_and_preprocess_image(img_path)
            if original_img is None:
                continue
            
            # Apply CLAHE enhancement
            enhanced_img, clahe_gray = apply_clahe_grayscale(original_img)
            
            processed_images.append({
                'original_path': img_path,
                'processed_image': enhanced_img,  # 3-channel CLAHE enhanced
                'clahe_gray': clahe_gray,        # Single channel grayscale
                'is_enhanced': True
            })
            
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            # Fallback to original image
            original_img = load_and_preprocess_image(img_path)
            if original_img is not None:
                processed_images.append({
                    'original_path': img_path,
                    'processed_image': original_img,
                    'clahe_gray': None,
                    'is_enhanced': False
                })
    
    print(f"✓ CLAHE enhancement completed for {dataset_name}: {len(processed_images)} images")
    return processed_images

def visualize_clahe_samples(processed_images, dataset_name, num_samples=5):
    """Visualize samples of CLAHE enhancement results"""
    print(f"Visualizing CLAHE enhancement for {dataset_name}...")
    
    samples = min(num_samples, len(processed_images))
    fig, axes = plt.subplots(samples, 3, figsize=(15, 5 * samples))
    
    if samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(samples):
        img_data = processed_images[i]
        
        # Load original for comparison
        original_img = load_and_preprocess_image(img_data['original_path'])
        
        # Original image
        axes[i, 0].imshow(original_img)
        axes[i, 0].set_title('Original Image')
        axes[i, 0].axis('off')
        
        # CLAHE enhanced grayscale (single channel)
        if img_data['clahe_gray'] is not None:
            axes[i, 1].imshow(img_data['clahe_gray'], cmap='gray')
            axes[i, 1].set_title('CLAHE Grayscale')
        else:
            axes[i, 1].imshow(original_img, cmap='gray')
            axes[i, 1].set_title('Original Grayscale')
        axes[i, 1].axis('off')
        
        # CLAHE enhanced RGB (3-channel)
        axes[i, 2].imshow(img_data['processed_image'])
        axes[i, 2].set_title('CLAHE Enhanced (RGB)')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, f'clahe_enhancement_{dataset_name}.png'), 
                dpi=150, bbox_inches='tight')

#### Disable Preprocessing Image

In [ ]:
##########################################################################

# Apply background removal to both datasets
#print("=== BACKGROUND REMOVAL AND THRESHOLDING ===")
#foreign_processed = apply_background_removal_to_dataset(foreign_valid, "foreign")
#ph_processed = apply_background_removal_to_dataset(ph_valid, "Philippines")

# Visualize results
#visualize_background_removal_samples(foreign_processed, "foreign")
#visualize_background_removal_samples(ph_processed, "Philippines")

#print("Background removal and thresholding completed.")

##########################################################################

##########################################################################

print("=== CLAHE ENHANCEMENT ===")
foreign_processed = apply_clahe_to_dataset(foreign_valid, "foreign")
ph_processed = apply_clahe_to_dataset(ph_valid, "Philippines")

# Visualize results
visualize_clahe_samples(foreign_processed, "foreign")
visualize_clahe_samples(ph_processed, "Philippines")

print("CLAHE enhancement completed.")

##########################################################################

##########################################################################

# Disabled Image Processing
# Create dummy processed images (just original images)
#foreign_processed = create_dummy_processed_images(foreign_valid)
#ph_processed = create_dummy_processed_images(ph_valid)

#print("Background removal step skipped - using original images")

##########################################################################

### 1.4 Annotate/Labeling of Images & 1.4 Generate CSV/JSON

In [ ]:
def create_annotation_format(filename, label, origin, unique_id, transformation=None):
    """Create annotation following the specified format"""
    base_name = f"{label}_{origin}_{unique_id:04d}"
    if transformation:
        base_name += f"_{transformation}"
    return f"{base_name}{Path(filename).suffix}"

def auto_detect_label(image_path):
    """Automatically detect label based on file path and name"""
    path_lower = image_path.lower()

    if 'blast' in path_lower or 'disease' in path_lower or 'infected' in path_lower:
        return 'LEAFBLAST'
    elif 'healthy' in path_lower or 'normal' in path_lower:
        return 'HEALTHY'
    else:
        return 'UNKNOWN'

def create_metadata(valid_images, invalid_images, dataset_name):
    """Create metadata with annotations for all images - WITHOUT BACKGROUND REMOVAL"""
    metadata = []
    unique_id = 1

    for img_data in valid_images:
        # FIX: Extract the path from the dictionary if it's a processed image
        if isinstance(img_data, dict):
            img_path = img_data['original_path']
        else:
            img_path = img_data

        label = auto_detect_label(img_path)
        origin = 'foreignI' if dataset_name == 'foreign' else 'LOCAL'

        annotated_name = create_annotation_format(
            Path(img_path).name,
            label,
            origin,
            unique_id
        )

        metadata.append({
            'original_path': img_path,
            'annotated_name': annotated_name,
            'label': label,
            'origin': origin,
            'unique_id': unique_id,
            'dataset': dataset_name,
            'is_background_removed': False,  # Always False now
            'has_background_mask': False     # Always False now
        })
        unique_id += 1

    # Save to CSV
    df = pd.DataFrame(metadata)
    csv_path = os.path.join(OUTPUT_BASE, f'{dataset_name}_metadata.csv')
    df.to_csv(csv_path, index=False)

    # Save to JSON
    json_path = os.path.join(OUTPUT_BASE, f'{dataset_name}_metadata.json')
    with open(json_path, 'w') as f:
        json_metadata = []
        for item in metadata:
            json_metadata.append({
                'original_path': item['original_path'],
                'annotated_name': item['annotated_name'],
                'label': item['label'],
                'origin': item['origin'],
                'unique_id': item['unique_id'],
                'dataset': item['dataset'],
                'is_background_removed': False,
                'has_background_mask': False
            })
        json.dump(json_metadata, f, indent=2)

    print(f"Saved {len(metadata)} records for {dataset_name} dataset")
    print(f"Background removal: DISABLED for all images")

    return df

# Remove the duplicate calls that were causing the error
# Create metadata for both datasets - USING ORIGINAL IMAGES (ONLY ONCE)
foreign_metadata = create_metadata(foreign_valid, foreign_invalid, 'foreign')
ph_metadata = create_metadata(ph_valid, ph_invalid, 'philippines')

print("Image annotation and metadata generation completed.")

### 1.5 Exploratory Data Analysis (EDA)

In [ ]:
def perform_eda(foreign_metadata, ph_metadata):
    """Perform combined EDA for both datasets - FIXED VERSION"""
    print(f"\nPerforming COMBINED EDA for both datasets...")

    # Combine both datasets (these are now DataFrames, not lists)
    combined_meta = pd.concat([foreign_metadata, ph_metadata], ignore_index=True)

    # Plot combined distribution
    plt.figure(figsize=(15, 6))

    # Plot 1: Combined origin distribution (pie chart)
    plt.subplot(1, 3, 1)
    origin_counts = combined_meta['origin'].value_counts()
    plt.pie(origin_counts.values, labels=origin_counts.index, autopct='%1.1f%%',
            colors=['lightblue', 'lightcoral'])
    plt.title('Combined Image Distribution by Origin')

    # Plot 2: Combined label distribution (bar chart)
    plt.subplot(1, 3, 2)
    label_counts = combined_meta['label'].value_counts()
    bars = plt.bar(range(len(label_counts)), label_counts.values,
                   color=['red', 'green', 'gray'])
    plt.xticks(range(len(label_counts)), label_counts.index, rotation=45, ha='right')
    plt.title('Combined Image Distribution by Label')

    # Add value labels on bars
    for i, bar in enumerate(bars):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom')

    # Plot 3: Background removal success - FIXED
    plt.subplot(1, 3, 3)
    if 'is_background_removed' in combined_meta.columns:
        bg_removal_counts = combined_meta['is_background_removed'].value_counts()

        # Handle cases where we might have only True or only False values
        true_count = bg_removal_counts.get(True, 0)
        false_count = bg_removal_counts.get(False, 0)

        # Create safe data for pie chart
        values = [true_count, false_count]
        labels = ['Background Removed', 'Original']
        colors = ['lightgreen', 'lightyellow']

        # Only plot if we have data
        if sum(values) > 0:
            plt.pie(values, labels=labels, autopct='%1.1f%%', colors=colors)
        else:
            plt.text(0.5, 0.5, 'No Background\nRemoval Data',
                    ha='center', va='center', transform=plt.gca().transAxes)

        plt.title('Background Removal Success Rate')
    else:
        plt.text(0.5, 0.5, 'Background Removal\nData Not Available',
                ha='center', va='center', transform=plt.gca().transAxes)
        plt.title('Background Removal Status')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'combined_data_distribution.png'), dpi=300, bbox_inches='tight')
    plt.show()

    # Print detailed statistics
    print("\n" + "="*50)
    print("COMBINED DATASET STATISTICS")
    print("="*50)

    print(f"\nTotal images: {len(combined_meta)}")
    print(f"foreigni images: {len(foreign_metadata)}")
    print(f"Philippines images: {len(ph_metadata)}")

    print("\nOverall class distribution:")
    print(combined_meta['label'].value_counts())

    print("\nforeigni dataset class distribution:")
    print(foreign_metadata['label'].value_counts())

    print("\nPhilippines dataset class distribution:")
    print(ph_metadata['label'].value_counts())

    # Background removal statistics
    if 'is_background_removed' in combined_meta.columns:
        bg_removed_total = combined_meta['is_background_removed'].sum()
        bg_removed_percentage = bg_removed_total / len(combined_meta) * 100
        print(f"\nBackground Removal Statistics:")
        print(f"Images with background removed: {bg_removed_total}/{len(combined_meta)} ({bg_removed_percentage:.1f}%)")

    # Calculate percentages
    print("\nPercentage distribution - Combined:")
    total = len(combined_meta)
    for label, count in combined_meta['label'].value_counts().items():
        print(f"  {label}: {count} ({count/total*100:.1f}%)")

    print("\nPercentage distribution - foreign:")
    total_bd = len(foreign_metadata)
    for label, count in foreign_metadata['label'].value_counts().items():
        print(f"  {label}: {count} ({count/total_bd*100:.1f}%)")

    print("\nPercentage distribution - Philippines:")
    total_ph = len(ph_metadata)
    for label, count in ph_metadata['label'].value_counts().items():
        print(f"  {label}: {count} ({count/total_ph*100:.1f}%)")

    # Return the combined statistics
    return {
        'combined_meta': combined_meta,
        'foreign_counts': foreign_metadata['label'].value_counts(),
        'ph_counts': ph_metadata['label'].value_counts(),
        'combined_counts': combined_meta['label'].value_counts()
    }

# Perform COMBINED EDA for both datasets
combined_stats = perform_eda(foreign_metadata, ph_metadata)

print("Combined Exploratory Data Analysis completed.")

### 1.6 Class Balance Auto

In [ ]:
def visualize_class_balance_comprehensive(foreign_metadata, ph_metadata, foreign_balanced, ph_balanced):
    """Comprehensive visualization of class balance before and after balancing"""
    print("Generating comprehensive class balance visualization...")

    # Create subplots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Before balancing - foreign
    bd_before_counts = foreign_metadata['label'].value_counts()
    axes[0, 0].pie(bd_before_counts.values, labels=bd_before_counts.index, autopct='%1.1f%%',
                   colors=['lightcoral', 'lightgreen', 'lightblue'])
    axes[0, 0].set_title('foreign - Before Balancing', fontsize=14, fontweight='bold')

    # After balancing - foreign
    bd_after_counts = foreign_balanced['label'].value_counts()
    axes[0, 1].pie(bd_after_counts.values, labels=bd_after_counts.index, autopct='%1.1f%%',
                   colors=['lightcoral', 'lightgreen', 'lightblue'])
    axes[0, 1].set_title('foreign - After Balancing', fontsize=14, fontweight='bold')

    # Before balancing - Philippines
    ph_before_counts = ph_metadata['label'].value_counts()
    axes[0, 2].pie(ph_before_counts.values, labels=ph_before_counts.index, autopct='%1.1f%%',
                   colors=['lightcoral', 'lightgreen', 'lightblue'])
    axes[0, 2].set_title('Philippines - Before Balancing', fontsize=14, fontweight='bold')

    # After balancing - Philippines
    ph_after_counts = ph_balanced['label'].value_counts()
    axes[1, 0].pie(ph_after_counts.values, labels=ph_after_counts.index, autopct='%1.1f%%',
                   colors=['lightcoral', 'lightgreen', 'lightblue'])
    axes[1, 0].set_title('Philippines - After Balancing', fontsize=14, fontweight='bold')

    # Bar chart comparison - foreign
    x = np.arange(len(bd_before_counts))
    width = 0.35
    axes[1, 1].bar(x - width/2, bd_before_counts.values, width, label='Before', alpha=0.7, color='red')
    axes[1, 1].bar(x + width/2, bd_after_counts.values, width, label='After', alpha=0.7, color='blue')
    axes[1, 1].set_xlabel('Classes')
    axes[1, 1].set_ylabel('Number of Images')
    axes[1, 1].set_title('foreign - Balance Comparison', fontsize=14, fontweight='bold')
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(bd_before_counts.index)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    # Bar chart comparison - Philippines
    x = np.arange(len(ph_before_counts))
    axes[1, 2].bar(x - width/2, ph_before_counts.values, width, label='Before', alpha=0.7, color='red')
    axes[1, 2].bar(x + width/2, ph_after_counts.values, width, label='After', alpha=0.7, color='blue')
    axes[1, 2].set_xlabel('Classes')
    axes[1, 2].set_ylabel('Number of Images')
    axes[1, 2].set_title('Philippines - Balance Comparison', fontsize=14, fontweight='bold')
    axes[1, 2].set_xticks(x)
    axes[1, 2].set_xticklabels(ph_before_counts.index)
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'comprehensive_class_balance.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # Print detailed statistics
    print("\n" + "="*60)
    print("CLASS BALANCE STATISTICS")
    print("="*60)

    # foreign stats
    bd_before_total = len(foreign_metadata)
    bd_after_total = len(foreign_balanced)
    bd_removed = bd_before_total - bd_after_total

    print(f"\nforeign Dataset:")
    print(f"  Before balancing: {bd_before_total} images")
    print(f"  After balancing:  {bd_after_total} images")
    print(f"  Removed samples:  {bd_removed} images")
    print(f"  Balance ratio: {bd_after_counts.min()}/{bd_after_counts.max()} "
          f"({bd_after_counts.min()/bd_after_counts.max()*100:.1f}%)")

    # Philippines stats
    ph_before_total = len(ph_metadata)
    ph_after_total = len(ph_balanced)
    ph_removed = ph_before_total - ph_after_total

    print(f"\nPhilippines Dataset:")
    print(f"  Before balancing: {ph_before_total} images")
    print(f"  After balancing:  {ph_after_total} images")
    print(f"  Removed samples:  {ph_removed} images")
    print(f"  Balance ratio: {ph_after_counts.min()}/{ph_after_counts.max()} "
          f"({ph_after_counts.min()/ph_after_counts.max()*100:.1f}%)")

    # Overall impact
    total_before = bd_before_total + ph_before_total
    total_after = bd_after_total + ph_after_total
    total_removed = total_before - total_after

    print(f"\nOverall Impact:")
    print(f"  Total before balancing: {total_before} images")
    print(f"  Total after balancing:  {total_after} images")
    print(f"  Total removed:          {total_removed} images")
    print(f"  Reduction: {total_removed/total_before*100:.1f}%")

In [ ]:
def balance_classes_undersample(metadata_df, dataset_name):
    """Balance classes by REDUCING majority class to match minority class"""
    print(f"Balancing classes for {dataset_name} dataset (undersampling)...")

    # Get class distribution
    label_counts = metadata_df['label'].value_counts()
    print(f"Current class distribution:")
    for label, count in label_counts.items():
        print(f"  {label}: {count} images")

    # Find the minority class and its count
    minority_label = label_counts.index[-1]  # Last one is the smallest
    minority_count = label_counts.iloc[-1]

    print(f"Minority class: {minority_label} with {minority_count} images")
    print(f"Target count for all classes: {minority_count} images")

    balanced_dfs = []

    for label in label_counts.index:
        class_df = metadata_df[metadata_df['label'] == label]
        current_count = len(class_df)

        if current_count > minority_count:
            # Need to undersample this class (reduce to minority count)
            print(f"Undersampling {label}: {current_count} -> {minority_count} (removing {current_count - minority_count})")

            # Randomly sample without replacement to reduce to minority count
            undersampled = class_df.sample(n=minority_count, replace=False, random_state=42)
            balanced_dfs.append(undersampled)

        elif current_count < minority_count:
            # This shouldn't happen if minority_count is truly the minimum
            print(f"Warning: {label} has {current_count} which is less than minority count {minority_count}")
            balanced_dfs.append(class_df)
        else:
            # Already at the target count
            print(f"Keeping {label}: {current_count} images (already balanced)")
            balanced_dfs.append(class_df)

    # Combine all balanced classes
    balanced_df = pd.concat(balanced_dfs, ignore_index=True)

    # Verify new distribution
    balanced_counts = balanced_df['label'].value_counts()
    print(f"Balanced class distribution:")
    for label, count in balanced_counts.items():
        print(f"  {label}: {count} images")

    # Calculate balance statistics
    original_total = len(metadata_df)
    balanced_total = len(balanced_df)
    removed_samples = original_total - balanced_total

    print(f"Balance summary:")
    print(f"  Original total: {original_total}")
    print(f"  Balanced total: {balanced_total}")
    print(f"  Removed samples: {removed_samples}")
    print(f"  Balance ratio: {balanced_counts.min()}/{balanced_counts.max()} "
          f"({balanced_counts.min()/balanced_counts.max()*100:.1f}%)")

    return balanced_df

# Balance both datasets using undersampling
print("\nBalancing foreign dataset...")
foreign_balanced = balance_classes_undersample(foreign_metadata, "foreign")

print("\nBalancing Philippines dataset...")
ph_balanced = balance_classes_undersample(ph_metadata, "Philippines")

# FIRST: Visualize comparison (original vs balanced)
print("Generating enhanced class balance visualization...")
visualize_class_balance_comprehensive(foreign_metadata, ph_metadata, foreign_balanced, ph_balanced)

# THEN: Update the metadata with balanced versions
foreign_metadata = foreign_balanced
ph_metadata = ph_balanced

print("\nClass balancing completed!")
print("="*60)

### 1.7 Dataset Splitting

In [ ]:
def split_datasets(foreign_df, ph_df):
    """Split datasets according to requirements"""
    print("Splitting datasets...")

    # Split foreign dataset: 80% training, 20% validation
    train_df, val_df = train_test_split(
        foreign_df,
        test_size=0.2,
        random_state=42,
        stratify=foreign_df['label']
    )

    # Philippines dataset for testing
    test_df = ph_df.copy()

    print(f"Training set (foreign): {len(train_df)} images")
    print(f"Validation set (foreign): {len(val_df)} images")
    print(f"Testing set (Philippines): {len(test_df)} images")

    # Save split datasets
    train_df.to_csv(os.path.join(OUTPUT_BASE, 'train_dataset.csv'), index=False)
    val_df.to_csv(os.path.join(OUTPUT_BASE, 'val_dataset.csv'), index=False)
    test_df.to_csv(os.path.join(OUTPUT_BASE, 'test_dataset.csv'), index=False)

    return train_df, val_df, test_df

# Split datasets
train_df, val_df, test_df = split_datasets(foreign_metadata, ph_metadata)

print("Dataset splitting completed.")

### 1.8 Dataset Augmentation

In [ ]:
# Only run augmentation if run_aug is True
if run_aug:
    import random

    print(f"Data augmentation ENABLED")
    print(f"Output folder set to: {OUTPUT_BASE}")

    def visualize_augmentation_samples(original_paths, augmented_data, processed_dict=None, num_samples=3):
        """Visualize original images and their augmented versions using processed images when available"""
        print(f"\nVisualizing augmentation samples...")

        # Get unique original images from augmented data
        unique_original_paths = list(set([item['original_path'] for item in augmented_data]))

        # Select random samples
        if len(unique_original_paths) < num_samples:
            num_samples = len(unique_original_paths)

        selected_paths = random.sample(unique_original_paths, num_samples)

        # Create figure
        fig, axes = plt.subplots(num_samples, 6, figsize=(20, 4 * num_samples))
        if num_samples == 1:
            axes = axes.reshape(1, -1)

        for i, original_path in enumerate(selected_paths):
            # Get original image - use processed if available, otherwise load original
            if processed_dict is not None and original_path in processed_dict:
                original_img = processed_dict[original_path]
                image_source = "Processed"
            else:
                original_img = load_and_preprocess_image(original_path)
                image_source = "Original"

            # Get augmented versions of this image
            aug_versions = [item for item in augmented_data if item['original_path'] == original_path]

            # Display original image
            axes[i, 0].imshow(original_img)
            axes[i, 0].set_title(f'{image_source} Image\n{Path(original_path).name}', fontsize=10)
            axes[i, 0].axis('off')

            # Display up to 5 augmented versions
            for j in range(min(5, len(aug_versions))):
                aug_img = aug_versions[j]['augmented_image']
                aug_type = aug_versions[j]['augmentation_type']

                axes[i, j+1].imshow(aug_img)
                axes[i, j+1].set_title(f'Augmented\n{aug_type}', fontsize=10)
                axes[i, j+1].axis('off')

            # If we have less than 5 augmentations, hide empty subplots
            for j in range(len(aug_versions) + 1, 6):
                axes[i, j].set_visible(False)

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_BASE, 'augmentation_samples.png'),
                    dpi=300, bbox_inches='tight')
        plt.show()

    def apply_data_augmentation_with_processed(train_df, augmentation_percentage=0.3, processed_dict=None):
        """Apply data augmentation using processed images when available"""
        print(f"\nApplying data augmentation ({augmentation_percentage*100}% increase)...")

        def adjust_brightness(image, factor):
            """Adjust image brightness"""
            return np.clip(image * factor, 0, 1)

        def adjust_contrast(image, factor):
            """Adjust image contrast"""
            mean = np.mean(image, axis=(0,1), keepdims=True)
            return np.clip((image - mean) * factor + mean, 0, 1)

        # Define all augmentation types
        augmentation_types = [
            ('flip_h', lambda img: np.fliplr(img)),
            ('flip_v', lambda img: np.flipud(img)),
            ('rot90', lambda img: np.rot90(img, 1)),
            ('rot180', lambda img: np.rot90(img, 2)),
            ('rot270', lambda img: np.rot90(img, 3)),
            ('bright_high', lambda img: adjust_brightness(img, 1.3)),
            ('bright_low', lambda img: adjust_brightness(img, 0.7)),
            ('contrast_high', lambda img: adjust_contrast(img, 1.5)),
            ('contrast_low', lambda img: adjust_contrast(img, 0.7)),
        ]

        print(f"Available augmentation types: {len(augmentation_types)}")
        print(f"Using processed images for augmentation: {processed_dict is not None}")

        augmented_data = []

        # Process each class separately
        for label in train_df['label'].unique():
            print(f"\nAugmenting {label} class...")
            class_images = train_df[train_df['label'] == label]
            original_count = len(class_images)

            # Calculate number of augmented samples needed
            samples_needed = max(
                len(augmentation_types),  # Minimum: one of each augmentation type
                int(original_count * augmentation_percentage)  # Percentage of original
            )

            print(f"  Original images: {original_count}")
            print(f"  Target augmented samples: {samples_needed}")

            # First pass: Ensure we have at least one of each augmentation type
            base_augmentations = []
            available_images = class_images.copy().reset_index(drop=True)

            for aug_idx, (aug_type_name, aug_func) in enumerate(augmentation_types):
                if aug_idx < len(available_images):
                    # Use a different image for each augmentation type
                    row = available_images.iloc[aug_idx]
                else:
                    # If we have more augmentation types than images, reuse images
                    row = available_images.sample(1).iloc[0]

                try:
                    # Use processed image if available, otherwise load original
                    if processed_dict is not None and row['original_path'] in processed_dict:
                        original_img = processed_dict[row['original_path']]
                    else:
                        original_img = load_and_preprocess_image(row['original_path'])

                    if original_img is None:
                        continue

                    aug_img = aug_func(original_img)

                    base_augmentations.append({
                        'original_path': row['original_path'],
                        'augmented_image': aug_img,
                        'augmentation_type': aug_type_name,
                        'label': row['label'],
                        'origin': row['origin'],
                        'unique_id': row['unique_id'],
                        'row_index': aug_idx
                    })

                except Exception as e:
                    print(f"Error in base augmentation for {row['original_path']}: {e}")

            # Second pass: Fill remaining needed samples
            additional_augmentations = []
            augment_count = len(base_augmentations)

            while augment_count < samples_needed:
                # Randomly select an image and augmentation
                random_row = class_images.sample(1).iloc[0]
                aug_type_name, aug_func = random.choice(augmentation_types)

                try:
                    # Use processed image if available
                    if processed_dict is not None and random_row['original_path'] in processed_dict:
                        original_img = processed_dict[random_row['original_path']]
                    else:
                        original_img = load_and_preprocess_image(random_row['original_path'])

                    if original_img is None:
                        continue

                    aug_img = aug_func(original_img)

                    additional_augmentations.append({
                        'original_path': random_row['original_path'],
                        'augmented_image': aug_img,
                        'augmentation_type': aug_type_name,
                        'label': random_row['label'],
                        'origin': random_row['origin'],
                        'unique_id': random_row['unique_id'],
                        'row_index': len(class_images) + augment_count
                    })

                    augment_count += 1
                    if augment_count % 10 == 0:
                        print(f"  Generated {augment_count}/{samples_needed} augmentations", end='\r')

                except Exception as e:
                    print(f"Error in additional augmentation: {e}")

            # Combine and create final entries
            all_augmentations = base_augmentations + additional_augmentations

            for i, aug_data in enumerate(all_augmentations):
                augmented_name = create_annotation_format(
                    Path(aug_data['original_path']).name,
                    aug_data['label'],
                    aug_data['origin'],
                    aug_data['unique_id'] * 1000 + i,
                    f"AUG_{aug_data['augmentation_type']}"
                )

                augmented_data.append({
                    'original_path': aug_data['original_path'],
                    'annotated_name': augmented_name,
                    'label': aug_data['label'],
                    'origin': aug_data['origin'],
                    'unique_id': aug_data['unique_id'] * 1000 + i,
                    'augmented_image': aug_data['augmented_image'],
                    'is_augmented': True,
                    'augmentation_type': aug_data['augmentation_type']
                })

            print(f"  ✓ Generated {len(all_augmentations)} augmentations for {label}")

        print(f"\nGenerated {len(augmented_data)} augmented images total")

        return augmented_data

    # Create processed image dictionary for augmentation
    foreign_processed_dict = {}
    if 'foreign_processed' in locals() and foreign_processed is not None:
        for item in foreign_processed:
            foreign_processed_dict[item['original_path']] = item['processed_image']
        print(f"Using {len(foreign_processed_dict)} processed images for augmentation")
    else:
        print("No processed images available - using original images for augmentation")

    # Store original training data
    original_train_size = len(train_df)
    original_train_df = train_df.copy()

    # Ensure original_train_df has is_augmented column set to False
    if 'is_augmented' not in original_train_df.columns:
        original_train_df['is_augmented'] = False
    else:
        original_train_df['is_augmented'] = original_train_df['is_augmented'].astype(bool)

    # Apply augmentation WITH processed images
    augmentation_percentage = 0.3  # 30% increase for both classes
    print(f"Using augmentation percentage: {augmentation_percentage*100}%")

    augmented_data = apply_data_augmentation_with_processed(
        train_df,
        augmentation_percentage,
        processed_dict=foreign_processed_dict
    )
    augmented_df = pd.DataFrame(augmented_data)

    # VISUALIZATION: Show augmentation samples using processed images when available
    visualize_augmentation_samples(
        original_train_df['original_path'].tolist(),
        augmented_data,
        processed_dict=foreign_processed_dict,
        num_samples=4
    )

    # Combine original and augmented data
    print(f"\nCombining datasets:")
    print(f"  Original training data: {len(original_train_df)} rows")
    print(f"  Augmented data: {len(augmented_df)} rows")

    # Ensure both DataFrames have compatible columns
    required_columns = ['original_path', 'annotated_name', 'label', 'origin', 'unique_id', 'is_augmented']

    # Make sure original_train_df has all required columns
    for col in required_columns:
        if col not in original_train_df.columns and col != 'is_augmented':
            original_train_df[col] = None

    # Make sure augmented_df has all required columns
    for col in required_columns:
        if col not in augmented_df.columns and col != 'is_augmented':
            augmented_df[col] = None

    # Now combine
    train_df = pd.concat([original_train_df, augmented_df], ignore_index=True)

    # Ensure is_augmented is boolean
    train_df['is_augmented'] = train_df['is_augmented'].astype(bool)

    print(f"  Final combined dataset: {len(train_df)} rows")
    print(f"  - Original samples: {(~train_df['is_augmented']).sum()}")
    print(f"  - Augmented samples: {train_df['is_augmented'].sum()}")

    # Save information about the augmentation
    augmentation_info = {
        'original_training_size': original_train_size,
        'augmented_samples': len(augmented_df),
        'final_training_size': len(train_df),
        'augmentation_percentage': augmentation_percentage,
        'augmentation_subfolder': OUTPUT_BASE,
        'used_processed_images': len(foreign_processed_dict) > 0
    }

    with open(os.path.join(OUTPUT_BASE, 'augmentation_info.json'), 'w') as f:
        json.dump(augmentation_info, f, indent=2)

    augmented_df.to_csv(os.path.join(OUTPUT_BASE, 'augmented_data.csv'), index=False)

    print(f"\nAugmentation Summary:")
    print(f"Original training set: {original_train_size} images")
    print(f"Augmented samples: {len(augmented_df)} images")
    print(f"Final training set: {len(train_df)} images")
    print(f"Augmentation increased dataset by {len(augmented_df)/original_train_size*100:.1f}%")
    print(f"Used processed images: {len(foreign_processed_dict) > 0}")
    print(f"Output folder: {OUTPUT_BASE}")

    print("Dataset augmentation phase completed.")

else:
    print("="*60)
    print("DATA AUGMENTATION SKIPPED")
    print("="*60)
    print("run_aug is set to False - proceeding without data augmentation")
    print(f"Using original training set: {len(train_df)} images")
    print("="*60)

## 2. CNN Training Module

### 2.0 Data Preparation Cell

In [ ]:
def prepare_training_data_orthogonal(foreign_metadata, ph_metadata, train_df=None, val_df=None, test_df=None,
                                   foreign_processed=None, ph_processed=None):
    """ORTHOGONAL VERSION: Handle both augmented data AND processed images independently"""

    # Independent decisions
    using_processed = foreign_processed is not None and ph_processed is not None
    using_augmented = train_df is not None and 'is_augmented' in train_df.columns and any(train_df['is_augmented'])

    print("="*60)
    print("ORTHOGONAL DATA PREPARATION")
    print("="*60)
    print(f"Image Preprocessing: {'CLAHE-Enhanced' if using_processed else 'Original Images'}")
    print(f"Data Augmentation: {'Enabled' if using_augmented else 'Disabled'}")

    # Create lookup dictionaries for processed images
    foreign_lookup = {}
    ph_lookup = {}

    if using_processed:
        for item in foreign_processed:
            foreign_lookup[item['original_path']] = item['processed_image']
        for item in ph_processed:
            ph_lookup[item['original_path']] = item['processed_image']
        print(f"Processed images available: foreign ({len(foreign_lookup)}), Philippines ({len(ph_lookup)})")

    def get_image(original_path, dataset_type="foreign"):
        """Get image - uses processed version if available, otherwise loads original"""
        if using_processed:
            lookup_dict = foreign_lookup if dataset_type == "foreign" else ph_lookup
            if original_path in lookup_dict:
                return lookup_dict[original_path]
        # Fallback to loading original image
        return load_and_preprocess_image(original_path)

    def safe_array_conversion(image_list):
        """Safely convert list of images to numpy array"""
        if not image_list:
            return np.array([])

        first_shape = image_list[0].shape
        all_same_shape = all(img.shape == first_shape for img in image_list)

        if all_same_shape:
            return np.array(image_list)
        else:
            print(f"Warning: Inconsistent image shapes. First: {first_shape}")
            shapes = [img.shape for img in image_list]
            shape_counts = {}
            for shape in shapes:
                shape_counts[shape] = shape_counts.get(shape, 0) + 1

            most_common_shape = max(shape_counts.items(), key=lambda x: x[1])[0]
            compatible_images = [img for img in image_list if img.shape == most_common_shape]
            print(f"Using {len(compatible_images)}/{len(image_list)} images with consistent shape")
            return np.array(compatible_images)

    # FIXED: Handle the case where we have processed images but no train_df (non-augmented)
    X_train = []
    y_train = []
    X_val = []
    y_val = []
    X_test = []
    y_test = []

    # CASE 1: We have train_df (augmented case OR non-augmented with DataFrame)
    if train_df is not None and val_df is not None:
        print(f"\nPreparing training data from DataFrame ({len(train_df)} rows)...")
        augmented_count = 0
        original_count = 0
        processed_count = 0

        for _, row in train_df.iterrows():
            try:
                # Handle augmented images (pre-computed)
                if using_augmented and row.get('is_augmented', False) and 'augmented_image' in row and row['augmented_image'] is not None:
                    aug_img = row['augmented_image']
                    if isinstance(aug_img, np.ndarray) and aug_img.shape == (224, 224, 3):
                        X_train.append(aug_img)
                        y_train.append(1 if row['label'] == 'LEAFBLAST' else 0)
                        augmented_count += 1
                else:
                    # Handle original images (load from path or use processed)
                    if 'original_path' in row and row['original_path'] is not None:
                        img_array = get_image(row['original_path'], "foreign")
                        if img_array is not None:
                            X_train.append(img_array)
                            y_train.append(1 if row['label'] == 'LEAFBLAST' else 0)
                            original_count += 1
                            if using_processed and row['original_path'] in foreign_lookup:
                                processed_count += 1
            except Exception as e:
                print(f"Error processing training row: {e}")
                continue

        print(f"  Training set composition:")
        print(f"    - Original images: {original_count} ({processed_count} from processing)")
        if using_augmented:
            print(f"    - Augmented images: {augmented_count}")
        print(f"    - Total: {len(X_train)}")

        # Prepare validation data
        print(f"\nPreparing validation data ({len(val_df)} rows)...")
        val_processed_count = 0

        for _, row in val_df.iterrows():
            if 'original_path' in row and row['original_path'] is not None:
                img_array = get_image(row['original_path'], "foreign")
                if img_array is not None:
                    X_val.append(img_array)
                    y_val.append(1 if row['label'] == 'LEAFBLAST' else 0)
                    if using_processed and row['original_path'] in foreign_lookup:
                        val_processed_count += 1

        print(f"  Validation set: {len(X_val)} images ({val_processed_count} from processing)")

    # CASE 2: No train_df available - use foreign_metadata directly (non-augmented case)
    elif foreign_metadata is not None:
        print(f"\nPreparing training data from metadata ({len(foreign_metadata)} rows)...")
        train_processed_count = 0

        # Split the foreign data for training and validation
        from sklearn.model_selection import train_test_split

        # Extract all foreign data
        X_foreign = []
        y_foreign = []

        for _, row in foreign_metadata.iterrows():
            if 'original_path' in row and row['original_path'] is not None:
                img_array = get_image(row['original_path'], "foreign")
                if img_array is not None:
                    X_foreign.append(img_array)
                    y_foreign.append(1 if row['label'] == 'LEAFBLAST' else 0)
                    if using_processed and row['original_path'] in foreign_lookup:
                        train_processed_count += 1

        print(f"  foreign dataset: {len(X_foreign)} images ({train_processed_count} from processing)")

        # Split into train and validation
        if len(X_foreign) > 0:
            X_train, X_val, y_train, y_val = train_test_split(
                X_foreign, y_foreign,
                test_size=0.2,
                random_state=42,
                stratify=y_foreign
            )
            print(f"  Training set: {len(X_train)} images")
            print(f"  Validation set: {len(X_val)} images")
        else:
            print("WARNING: No training data found!")

    else:
        print("WARNING: No training data source available!")

    # Prepare test data
    print(f"\nPreparing test data...")
    test_processed_count = 0

    # Try test_df first, then fall back to ph_metadata
    if test_df is not None:
        print(f"  From test_df ({len(test_df)} rows)...")
        for _, row in test_df.iterrows():
            if 'original_path' in row and row['original_path'] is not None:
                img_array = get_image(row['original_path'], "philippines")
                if img_array is not None:
                    X_test.append(img_array)
                    y_test.append(1 if row['label'] == 'LEAFBLAST' else 0)
                    if using_processed and row['original_path'] in ph_lookup:
                        test_processed_count += 1
    elif ph_metadata is not None:
        print(f"  From ph_metadata ({len(ph_metadata)} rows)...")
        for _, row in ph_metadata.iterrows():
            if 'original_path' in row and row['original_path'] is not None:
                img_array = get_image(row['original_path'], "philippines")
                if img_array is not None:
                    X_test.append(img_array)
                    y_test.append(1 if row['label'] == 'LEAFBLAST' else 0)
                    if using_processed and row['original_path'] in ph_lookup:
                        test_processed_count += 1

    print(f"  Test set: {len(X_test)} images ({test_processed_count} from processing)")

    # Convert to arrays
    X_train = safe_array_conversion(X_train)
    y_train = np.array(y_train)
    X_val = safe_array_conversion(X_val)
    y_val = np.array(y_val)
    X_test = safe_array_conversion(X_test)
    y_test = np.array(y_test)

    print(f"\nFinal dataset shapes:")
    print(f"Training: {X_train.shape}, {y_train.shape}")
    print(f"Validation: {X_val.shape}, {y_val.shape}")
    print(f"Test: {X_test.shape}, {y_test.shape}")

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
using_augmented = 'train_df' in locals() and 'is_augmented' in train_df.columns and any(train_df['is_augmented'])
using_processed = 'foreign_processed' in locals() and 'ph_processed' in locals() and foreign_processed is not None

print("="*60)
print("DATA SOURCE ANALYSIS")
print("="*60)
print(f"Augmented data available: {using_augmented}")
print(f"Processed images available: {using_processed}")

if using_augmented and using_processed:
    # CASE 1: BOTH augmented data AND processed images
    print("=== USING AUGMENTED DATA WITH PROCESSED IMAGES ===")
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_training_data_orthogonal(
        foreign_metadata, ph_metadata, train_df, val_df, test_df,
        foreign_processed=foreign_processed, ph_processed=ph_processed
    )

elif using_augmented and not using_processed:
    # CASE 2: Only augmented data (no processed images)
    print("=== USING AUGMENTED DATA WITH ORIGINAL IMAGES ===")
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_training_data_orthogonal(
        foreign_metadata, ph_metadata, train_df, val_df, test_df
    )

elif not using_augmented and using_processed:
    # CASE 3: Only processed images (no augmentation)
    print("=== USING PROCESSED IMAGES (NO AUGMENTATION) ===")
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_training_data_orthogonal(
        foreign_metadata, ph_metadata,
        foreign_processed=foreign_processed, ph_processed=ph_processed
    )

else:
    # CASE 4: Neither - use original images only
    print("=== USING ORIGINAL IMAGES (NO AUGMENTATION) ===")
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_training_data_orthogonal(
        foreign_metadata, ph_metadata
    )

In [ ]:
# Check dataset sizes and balance
print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Training class distribution: {np.unique(y_train, return_counts=True)}")
print(f"Validation class distribution: {np.unique(y_val, return_counts=True)}")

# Check if data is shuffled properly
print(f"First 10 training labels: {y_train[:10]}")
print(f"First 10 validation labels: {y_val[:10]}")

# Check if shape is correct
print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")
print(f"Number of training samples: {len(X_train)}")
print(f"Number of validation samples: {len(X_val)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Steps per epoch: {len(X_train) // BATCH_SIZE}")

#### Debug

In [ ]:
def debug_training_speed(X_train, X_val, y_train, y_val):
    """Debug why training speed differs between augmented vs non-augmented"""
    print("\n" + "="*50)
    print("TRAINING SPEED DIAGNOSTICS")
    print("="*50)

    # Check data sources
    print(f"Training samples: {len(X_train)}")
    print(f"X_train type: {type(X_train)}")
    print(f"X_train dtype: {X_train.dtype}")
    print(f"X_train shape: {X_train.shape}")

    # Check if data is already normalized
    print(f"Data range: [{X_train.min():.3f}, {X_train.max():.3f}]")

    # Check memory usage
    import sys
    memory_mb = sys.getsizeof(X_train) / (1024 * 1024)
    print(f"Training data memory: {memory_mb:.2f} MB")

    # Check data loading source
    if hasattr(X_train, 'flags') and hasattr(X_train.flags, 'writeable'):
        print(f"Data is in memory: {X_train.flags.writeable}")

    return memory_mb

# Add this before training
memory_usage = debug_training_speed(X_train, X_val, y_train, y_val)

### 2.1 VGG16 Baseline Model

In [ ]:
# 2.1 VGG16 HYPERPARAMETER TUNING & FINE-TUNING (REGULARIZED)
# =============================================================================
# UPDATED STRATEGY: AGGRESSIVE REGULARIZATION
# The previous run showed that the CNN Head was overfitting the source domain
# (foreign) while PLSR generalized well. This updated code adds L2 Regularization
# and restricts model complexity to force the CNN to learn robust, general features.
# =============================================================================

import keras_tuner as kt
from tensorflow.keras import regularizers

print("="*60)
print("2.1 VGG16 TUNING & FINE-TUNING (WITH L2 REGULARIZATION)")
print("="*60)

# -----------------------------------------------------------------------------
# 1. HYPERMODEL BUILDER FUNCTION
# -----------------------------------------------------------------------------
def build_hyper_vgg16(hp):
    """
    Builds a VGG16 hypermodel with L2 Regularization to prevent overfitting.
    """
    # 1. Base Model (VGG16)
    base_model = VGG16(
        include_top=False,
        weights='imagenet',
        input_shape=(TARGET_SIZE[0], TARGET_SIZE[1], 3)
    )
    base_model.trainable = False # Freeze base for initial search

    # 2. Sequential Model
    model = models.Sequential()
    model.add(base_model)
    model.add(layers.GlobalAveragePooling2D())

    # --- Hyperparameter Tuning (Regularized Head) ---

    # Tune L2 Regularization Factor (Critical for Generalization)
    # This forces weights to be small, preventing memorization
    hp_l2 = hp.Choice('l2_rate', values=[1e-3, 1e-4])

    # Tune Number of Neurons (Reduced max capacity to prevent overfitting)
    hp_units = hp.Int('dense_units', min_value=64, max_value=256, step=64)

    model.add(layers.Dense(
        units=hp_units,
        activation='relu',
        kernel_regularizer=regularizers.l2(hp_l2) # <--- Added L2 Regularization
    ))

    # Tune Dropout (Higher minimum to force robustness)
    hp_dropout = hp.Float('dropout_rate', min_value=0.4, max_value=0.7, step=0.1)
    model.add(layers.Dropout(rate=hp_dropout))

    # Output Layer
    model.add(layers.Dense(1, activation='sigmoid'))

    # 3. Compile
    hp_lr = hp.Choice('learning_rate', values=[1e-3, 5e-4])

    model.compile(
        optimizer=optimizers.Adam(learning_rate=hp_lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# -----------------------------------------------------------------------------
# 2. HYPERPARAMETER SEARCH
# -----------------------------------------------------------------------------
print("\n[Phase 1] Starting Regularized Search...")

tuner = kt.Hyperband(
    build_hyper_vgg16,
    objective='val_accuracy',
    max_epochs=15,
    factor=3,
    directory=os.path.join(OUTPUT_BASE, 'keras_tuner_regularized'),
    project_name='vgg16_reg_opt'
)

# Early stopping
tuner_callbacks = [EarlyStopping(monitor='val_loss', patience=4)]

tuner.search(
    X_train, y_train,
    epochs=5,
    validation_data=(X_val, y_val),
    callbacks=tuner_callbacks,
    verbose=1
)

# Get the best model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
baseline_model = tuner.get_best_models(num_models=1)[0]

print(f"\n[Result] Best Regularized Hyperparameters:")
print(f" - Dense Units: {best_hps.get('dense_units')}")
print(f" - Dropout Rate: {best_hps.get('dropout_rate')}")
print(f" - L2 Regularization: {best_hps.get('l2_rate')}")
print(f" - Learning Rate: {best_hps.get('learning_rate')}")

# -----------------------------------------------------------------------------
# 3. FINE-TUNING (CONSERVATIVE)
# -----------------------------------------------------------------------------
print("\n[Phase 2] Starting Conservative Fine-Tuning...")

# Unfreeze the VGG16 base
vgg_base = baseline_model.get_layer('vgg16')
vgg_base.trainable = True

# Freeze all layers EXCEPT the last block (Block 5)
fine_tune_at = 15
for layer in vgg_base.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with a VERY LOW learning rate
# We use a slightly smaller LR here to be safer with the L2 regularization
baseline_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Fine-tuning callbacks
ft_callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4, min_lr=1e-7, verbose=1)
]

# Train
history_fine_tune = baseline_model.fit(
    X_train, y_train,
    epochs=5,
    validation_data=(X_val, y_val),
    callbacks=ft_callbacks,
    verbose=1
)

# -----------------------------------------------------------------------------
# 4. SAVE RESULTS
# -----------------------------------------------------------------------------
baseline_history = history_fine_tune
feature_extractor = baseline_model.get_layer('vgg16')
baseline_loss, baseline_accuracy = baseline_model.evaluate(X_val, y_val, verbose=0)

# Save to disk
model_save_path = os.path.join(OUTPUT_BASE, 'baseline_vgg16_regularized.pkl')
joblib.dump(baseline_model, model_save_path)
joblib.dump(history_fine_tune.history, os.path.join(OUTPUT_BASE, 'baseline_history.pkl'))

print(f"\n[Completed] Final Validated Accuracy: {baseline_accuracy:.4f}")

In [ ]:
# 2.1.1 Feature Extraction Function
def extract_features_correct(baseline_model, x_data):
    """
    Extracts features from the baseline VGG16 model.

    This function accesses the first layer (index 0) of the sequential
    'baseline_model', which IS the VGG16 feature extractor base model,
    and calls .predict() on it directly.
    """
    print(f"Extracting features from {x_data.shape[0]} images...")

    # The 'baseline_model' is Sequential, and its first layer (index 0)
    # is the VGG16 base model itself. We can just use it directly.
    feature_extractor_model = baseline_model.layers[0]

    # Predict to get the features
    print(f"Using model layer: {feature_extractor_model.name}")
    features = feature_extractor_model.predict(x_data, batch_size=BATCH_SIZE, verbose=1)

    print(f"✓ Features extracted, shape: {features.shape}")
    return features

print("Feature extraction function 'extract_features_correct' defined.")

### 2.2 VGG16 + PLSR with Thresholding

In [ ]:
# 2.2 VGG16 + PLSR with Thresholding
print("="*60)
print("2.2 VGG16 + PLSR WITH THRESHOLDING")
print("="*60)

def train_plsr_model(baseline_model, X_train, X_val, y_train, y_val):
    """Train VGG16 + PLSR model with feature extraction - FIXED"""
    print("Extracting features for PLSR...")

    # Extract features using the corrected function
    train_features = extract_features_correct(baseline_model, X_train)
    val_features = extract_features_correct(baseline_model, X_val)

    print(f"Feature shapes - Train: {train_features.shape}, Val: {val_features.shape}")

    # If features are 4D (from conv layer), flatten them
    if len(train_features.shape) > 2:
        train_features = train_features.reshape(train_features.shape[0], -1)
        val_features = val_features.reshape(val_features.shape[0], -1)
        print(f"Flattened feature shapes - Train: {train_features.shape}, Val: {val_features.shape}")

    # Normalize features
    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features)
    val_features_scaled = scaler.transform(val_features)

    # Train PLSR
    print("Training PLSR model...")
    start_time = time.time()

    plsr = PLSRegression(n_components=50, max_iter=1000)
    plsr.fit(train_features_scaled, y_train)

    training_time = time.time() - start_time
    print(f"✓ PLSR training completed in {training_time:.2f} seconds")

    # Predict
    y_pred_proba_plsr = plsr.predict(val_features_scaled)
    y_pred_plsr = (y_pred_proba_plsr > 0.5).astype(int).flatten()

    plsr_accuracy = accuracy_score(y_val, y_pred_plsr)
    plsr_precision = precision_score(y_val, y_pred_plsr, zero_division=0)
    plsr_recall = recall_score(y_val, y_pred_plsr, zero_division=0)
    plsr_f1 = f1_score(y_val, y_pred_plsr, zero_division=0)

    print(f"✓ PLSR Model Evaluation:")
    print(f" Validation Accuracy: {plsr_accuracy:.4f}")
    print(f" Precision: {plsr_precision:.4f}")
    print(f" Recall: {plsr_recall:.4f}")
    print(f" F1-Score: {plsr_f1:.4f}")

    # Save PLSR model and scaler as pickle
    plsr_model_path = os.path.join(OUTPUT_BASE, 'plsr_model.pkl')
    plsr_scaler_path = os.path.join(OUTPUT_BASE, 'plsr_scaler.pkl')

    joblib.dump(plsr, plsr_model_path)
    joblib.dump(scaler, plsr_scaler_path)

    print(f"✓ PLSR model saved as: {plsr_model_path}")
    print(f"✓ PLSR scaler saved as: {plsr_scaler_path}")

    return plsr, scaler, plsr_accuracy, plsr_precision, plsr_recall, plsr_f1

# Train PLSR model
plsr_model, plsr_scaler, plsr_accuracy, plsr_precision, plsr_recall, plsr_f1 = train_plsr_model(
    baseline_model, X_train, X_val, y_train, y_val
)

# Store PLSR results
plsr_results = {
    'model': plsr_model,
    'scaler': plsr_scaler,
    'val_accuracy': plsr_accuracy,
    'precision': plsr_precision,
    'recall': plsr_recall,
    'f1_score': plsr_f1,
    'feature_extractor': baseline_model
}

print("2.2 VGG16 + PLSR with Thresholding - COMPLETED\n")

### 2.3 VGG16 + XGBoost Classifier

In [ ]:
# 2.3 VGG16 + XGBoost
print("="*60)
print("2.3 VGG16 + XGBOOST")
print("="*60)

def train_xgboost_model(baseline_model, X_train, X_val, y_train, y_val):
    """Train VGG16 + XGBoost model with feature extraction - FIXED"""
    if not XGB_AVAILABLE:
        print("XGBoost not available. Skipping...")
        return None, None, 0, 0, 0, 0

    print("Extracting features for XGBoost...")

    # Extract features using the corrected function
    train_features = extract_features_correct(baseline_model, X_train)
    val_features = extract_features_correct(baseline_model, X_val)

    print(f"Feature shapes - Train: {train_features.shape}, Val: {val_features.shape}")

    # If features are 4D (from conv layer), flatten them
    if len(train_features.shape) > 2:
        train_features = train_features.reshape(train_features.shape[0], -1)
        val_features = val_features.reshape(val_features.shape[0], -1)
        print(f"Flattened feature shapes - Train: {train_features.shape}, Val: {val_features.shape}")

    # Normalize features
    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features)
    val_features_scaled = scaler.transform(val_features)

    # Train XGBoost with progress tracking
    print("Training XGBoost model...")
    start_time = time.time()

    # Create XGBoost model
    # FIX 1: Moved 'early_stopping_rounds' from .fit() to the constructor.
    # This prevents the TypeError.
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        objective='binary:logistic',
        random_state=42,
        eval_metric='logloss',
        early_stopping_rounds=10 # <-- MOVED HERE
    )

    # Train with early stopping - .fit() no longer has the problem argument
    xgb_model.fit(
        train_features_scaled,
        y_train,
        eval_set=[(val_features_scaled, y_val)],
        verbose=True
    )

    training_time = time.time() - start_time
    print(f"✓ XGBoost training completed in {training_time:.2f} seconds")

    # Predict
    y_pred_xgb = xgb_model.predict(val_features_scaled)

    xgb_accuracy = accuracy_score(y_val, y_pred_xgb)
    xgb_precision = precision_score(y_val, y_pred_xgb, zero_division=0)
    xgb_recall = recall_score(y_val, y_pred_xgb, zero_division=0)
    xgb_f1 = f1_score(y_val, y_pred_xgb, zero_division=0)

    print(f"✓ XGBoost Model Evaluation:")
    print(f" Validation Accuracy: {xgb_accuracy:.4f}")
    print(f" Precision: {xgb_precision:.4f}")
    print(f" Recall: {xgb_recall:.4f}")
    print(f" F1-Score: {xgb_f1:.4f}")

    return xgb_model, scaler, xgb_accuracy, xgb_precision, xgb_recall, xgb_f1

# Alternative version if the above still doesn't work
def train_xgboost_model_simple(baseline_model, X_train, X_val, y_train, y_val):
    """Simplified XGBoost training without early stopping"""
    if not XGB_AVAILABLE:
        print("XGBoost not available. Skipping...")
        return None, None, 0, 0, 0, 0

    print("Extracting features for XGBoost...")

    # Extract features using the corrected function
    train_features = extract_features_correct(baseline_model, X_train)
    val_features = extract_features_correct(baseline_model, X_val)

    print(f"Feature shapes - Train: {train_features.shape}, Val: {val_features.shape}")

    # FIX 2: Added the missing feature-flattening block to the simple
    # fallback function. This prevents the ValueError.
    if len(train_features.shape) > 2:
        train_features = train_features.reshape(train_features.shape[0], -1)
        val_features = val_features.reshape(val_features.shape[0], -1)
        print(f"Flattened feature shapes - Train: {train_features.shape}, Val: {val_features.shape}")

    # Normalize features
    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features)
    val_features_scaled = scaler.transform(val_features)

    # Train XGBoost with progress tracking
    print("Training XGBoost model (simple)...")
    start_time = time.time()

    # Create XGBoost model
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        objective='binary:logistic',
        random_state=42,
        verbosity=1 # Show training progress
    )

    # Simple fit without early stopping
    xgb_model.fit(train_features_scaled, y_train)

    training_time = time.time() - start_time
    print(f"✓ XGBoost training completed in {training_time:.2f} seconds")

    # Predict
    y_pred_xgb = xgb_model.predict(val_features_scaled)

    xgb_accuracy = accuracy_score(y_val, y_pred_xgb)
    xgb_precision = precision_score(y_val, y_pred_xgb, zero_division=0)
    xgb_recall = recall_score(y_val, y_pred_xgb, zero_division=0)
    xgb_f1 = f1_score(y_val, y_pred_xgb, zero_division=0)

    print(f"✓ XGBoost Model Evaluation:")
    print(f" Validation Accuracy: {xgb_accuracy:.4f}")
    print(f" Precision: {xgb_precision:.4f}")
    print(f" Recall: {xgb_recall:.4f}")
    print(f" F1-Score: {xgb_f1:.4f}")

    # Save XGBoost model and scaler as pickle
    if xgb_model is not None:
        xgb_model_path = os.path.join(OUTPUT_BASE, 'xgboost_model.pkl')
        xgb_scaler_path = os.path.join(OUTPUT_BASE, 'xgboost_scaler.pkl')

        joblib.dump(xgb_model, xgb_model_path)
        joblib.dump(scaler, xgb_scaler_path)

        print(f"✓ XGBoost model saved as: {xgb_model_path}")
        print(f"✓ XGBoost scaler saved as: {xgb_scaler_path}")

    return xgb_model, scaler, xgb_accuracy, xgb_precision, xgb_recall, xgb_f1

# Try the first version, if it fails, use the simple version
try:
    # Train XGBoost model with early stopping
    print("Attempting XGBoost training with early stopping...")
    xgb_model, xgb_scaler, xgb_accuracy, xgb_precision, xgb_recall, xgb_f1 = train_xgboost_model(
        baseline_model, X_train, X_val, y_train, y_val
    )
except TypeError as e:
    print(f"Early stopping version failed: {e}")
    print("Trying simple version without early stopping...")
    xgb_model, xgb_scaler, xgb_accuracy, xgb_precision, xgb_recall, xgb_f1 = train_xgboost_model_simple(
        baseline_model, X_train, X_val, y_train, y_val
    )

# Store XGBoost results
xgb_results = {
    'model': xgb_model,
    'scaler': xgb_scaler,
    'val_accuracy': xgb_accuracy,
    'precision': xgb_precision,
    'recall': xgb_recall,
    'f1_score': f1_score,
    'feature_extractor': baseline_model
}

print("2.3 VGG16 + XGBoost - COMPLETED\n")

## 3. Validation Module

### 3.1 Model run with Validation set

In [ ]:
def validate_all_models(baseline_model, plsr_model, xgb_model, X_val, y_val):
    """Comprehensive validation of all trained models"""
    print("="*60)
    print("VALIDATION MODULE - MODEL EVALUATION")
    print("="*60)

    validation_results = {}

    # 1. Baseline VGG16 Validation
    print("\n1. Validating Baseline VGG16 Model...")
    baseline_val_proba = baseline_model.predict(X_val, verbose=0)
    baseline_val_pred = (baseline_val_proba > 0.5).astype(int).flatten()

    baseline_metrics = {
        'accuracy': accuracy_score(y_val, baseline_val_pred),
        'precision': precision_score(y_val, baseline_val_pred, zero_division=0),
        'recall': recall_score(y_val, baseline_val_pred, zero_division=0),
        'f1_score': f1_score(y_val, baseline_val_pred, zero_division=0),
        'predictions': baseline_val_pred,
        'probabilities': baseline_val_proba.flatten()
    }

    # Calculate specificity
    tn, fp, fn, tp = confusion_matrix(y_val, baseline_val_pred).ravel()
    baseline_metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0

    validation_results['baseline'] = baseline_metrics

    print(f"  ✓ Accuracy: {baseline_metrics['accuracy']:.4f}")
    print(f"  ✓ Precision: {baseline_metrics['precision']:.4f}")
    print(f"  ✓ Recall: {baseline_metrics['recall']:.4f}")
    print(f"  ✓ F1-Score: {baseline_metrics['f1_score']:.4f}")
    print(f"  ✓ Specificity: {baseline_metrics['specificity']:.4f}")

    # 2. PLSR Model Validation
    print("\n2. Validating PLSR Model...")
    # Extract features for validation set
    val_features_plsr = extract_features_correct(baseline_model, X_val)
    if len(val_features_plsr.shape) > 2:
        val_features_plsr = val_features_plsr.reshape(val_features_plsr.shape[0], -1)

    val_features_plsr_scaled = plsr_scaler.transform(val_features_plsr)
    plsr_val_pred_proba = plsr_model.predict(val_features_plsr_scaled)
    plsr_val_pred = (plsr_val_pred_proba > 0.5).astype(int).flatten()

    plsr_metrics = {
        'accuracy': accuracy_score(y_val, plsr_val_pred),
        'precision': precision_score(y_val, plsr_val_pred, zero_division=0),
        'recall': recall_score(y_val, plsr_val_pred, zero_division=0),
        'f1_score': f1_score(y_val, plsr_val_pred, zero_division=0),
        'predictions': plsr_val_pred,
        'probabilities': plsr_val_pred_proba.flatten()
    }

    tn, fp, fn, tp = confusion_matrix(y_val, plsr_val_pred).ravel()
    plsr_metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0

    validation_results['plsr'] = plsr_metrics

    print(f"  ✓ Accuracy: {plsr_metrics['accuracy']:.4f}")
    print(f"  ✓ Precision: {plsr_metrics['precision']:.4f}")
    print(f"  ✓ Recall: {plsr_metrics['recall']:.4f}")
    print(f"  ✓ F1-Score: {plsr_metrics['f1_score']:.4f}")
    print(f"  ✓ Specificity: {plsr_metrics['specificity']:.4f}")

    # 3. XGBoost Model Validation (if available)
    if XGB_AVAILABLE and xgb_model is not None:
        print("\n3. Validating XGBoost Model...")
        # Extract features for validation set
        val_features_xgb = extract_features_correct(baseline_model, X_val)
        if len(val_features_xgb.shape) > 2:
            val_features_xgb = val_features_xgb.reshape(val_features_xgb.shape[0], -1)

        val_features_xgb_scaled = xgb_scaler.transform(val_features_xgb)
        xgb_val_pred = xgb_model.predict(val_features_xgb_scaled)
        xgb_val_pred_proba = xgb_model.predict_proba(val_features_xgb_scaled)[:, 1]

        xgb_metrics = {
            'accuracy': accuracy_score(y_val, xgb_val_pred),
            'precision': precision_score(y_val, xgb_val_pred, zero_division=0),
            'recall': recall_score(y_val, xgb_val_pred, zero_division=0),
            'f1_score': f1_score(y_val, xgb_val_pred, zero_division=0),
            'predictions': xgb_val_pred,
            'probabilities': xgb_val_pred_proba
        }

        tn, fp, fn, tp = confusion_matrix(y_val, xgb_val_pred).ravel()
        xgb_metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0

        validation_results['xgboost'] = xgb_metrics

        print(f"  ✓ Accuracy: {xgb_metrics['accuracy']:.4f}")
        print(f"  ✓ Precision: {xgb_metrics['precision']:.4f}")
        print(f"  ✓ Recall: {xgb_metrics['recall']:.4f}")
        print(f"  ✓ F1-Score: {xgb_metrics['f1_score']:.4f}")
        print(f"  ✓ Specificity: {xgb_metrics['specificity']:.4f}")

    # Create validation results visualization
    plot_validation_results(validation_results)

    # Save validation results as pickle
    validation_results_path = os.path.join(OUTPUT_BASE, 'validation_results.pkl')
    joblib.dump(validation_results, validation_results_path)
    print(f"✓ Validation results saved as: {validation_results_path}")

    return validation_results

def plot_validation_results(validation_results):
    """Plot comprehensive validation results"""
    models = list(validation_results.keys())
    metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'specificity']

    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()

    # Plot 1: All metrics comparison
    x = np.arange(len(models))
    width = 0.15

    for i, metric in enumerate(metrics):
        values = [validation_results[model][metric] for model in models]
        axes[0].bar(x + (i-2)*width, values, width, label=metric.capitalize(), alpha=0.8)

    axes[0].set_xlabel('Models')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Validation Metrics Comparison')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([model.upper() for model in models])
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(0, 1)

    # Plot 2: Confusion matrices
    for idx, model in enumerate(models[:3]):  # Show first 3 models
        if idx < 3:  # Ensure we don't exceed subplot count
            cm = confusion_matrix(y_val, validation_results[model]['predictions'])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx+1],
                       xticklabels=['Healthy', 'Blast'],
                       yticklabels=['Healthy', 'Blast'])
            axes[idx+1].set_title(f'{model.upper()} Confusion Matrix')
            axes[idx+1].set_xlabel('Predicted')
            axes[idx+1].set_ylabel('Actual')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'validation_results_comprehensive.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # Print detailed classification reports
    print("\n" + "="*50)
    print("DETAILED CLASSIFICATION REPORTS")
    print("="*50)

    for model in models:
        print(f"\n{model.upper()} Classification Report:")
        print(classification_report(y_val, validation_results[model]['predictions'],
                                  target_names=['Healthy', 'Leaf Blast']))

# Run validation
print("Starting comprehensive model validation...")
validation_results = validate_all_models(baseline_model, plsr_model, xgb_model, X_val, y_val)
print("Validation module completed successfully!")

### 3.2 Explainability Analysis

In [ ]:
def enhanced_gradcam_heatmap(img_array, model, last_conv_layer_name=None):
    try:
        # 1. Isolate the VGG16 base (First layer of your Sequential model)
        vgg_base = model.layers[0]
        
        # 2. Auto-detect the last CONVOLUTIONAL layer (skipping Pooling layers)
        if last_conv_layer_name is None:
            for layer in reversed(vgg_base.layers):
                # Check if it is a Conv2D layer (has 4D output and weights)
                if 'conv' in layer.name or isinstance(layer, tf.keras.layers.Conv2D):
                    last_conv_layer_name = layer.name
                    break
        
        # 3. Construct a sub-model for the VGG base
        # We need TWO outputs: The specific Conv layer (for heat) and the Final Base output (for flow)
        base_multi_out_model = tf.keras.models.Model(
            inputs=vgg_base.input,
            outputs=[vgg_base.get_layer(last_conv_layer_name).output, vgg_base.output]
        )

        # 4. Run the Gradient Calculation
        with tf.GradientTape() as tape:
            # Get the conv features and the base output
            conv_outputs, base_outputs = base_multi_out_model(img_array)
            
            # Manually pass the base output through the rest of your classifier (The "Head")
            # This connects the VGG base to your final Dense prediction
            preds = base_outputs
            for layer in model.layers[1:]: # Skip the first layer (vgg_base)
                preds = layer(preds)
            
            # Get the score for the top predicted class
            top_pred_index = tf.argmax(preds[0])
            loss = preds[:, top_pred_index]

        # 5. Calculate Gradients
        grads = tape.gradient(loss, conv_outputs)
        
        # 6. Pool Gradients (GAP)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

        # 7. Generate Heatmap
        conv_outputs = conv_outputs[0]
        heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)

        # 8. Clean Up Heatmap
        heatmap = heatmap.numpy()
        heatmap = np.maximum(heatmap, 0) # ReLU (Remove negative values)
        
        # CRITICAL: Noise Gate to "Specificially Highlight Blast"
        # If the heatmap is too fuzzy, we zero out weak activations
        if np.max(heatmap) > 0:
            heatmap /= np.max(heatmap) # Normalize 0-1
            
            # Optional: Threshold to remove background noise (e.g., weak activations < 0.4)
            # This makes the "spots" pop out more
            heatmap[heatmap < 0.3] = 0 
            
        return heatmap

    except Exception as e:
        print(f"Grad-CAM Error: {e}")
        return improved_fallback_heatmap(img_array, model)

def improved_fallback_heatmap(img_array, model):
    """Fallback to edge detection if Grad-CAM completely fails"""
    try:
        img_uint8 = (img_array[0] * 255).astype(np.uint8)
        img_gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
        grad_x = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=5)
        grad_y = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=5)
        magnitude = np.sqrt(grad_x**2 + grad_y**2)
        magnitude = cv2.GaussianBlur(magnitude, (15, 15), 3)
        heatmap = cv2.resize(magnitude, (7, 7))
        heatmap = np.power(heatmap, 0.6)
        if np.max(heatmap) > 0: heatmap /= np.max(heatmap)
        return heatmap
    except:
        return np.zeros((7, 7))

def apply_gradcam_validation(X_val, y_val, model, n_samples=6):
    """Apply Grad-CAM to validation samples and display results"""
    print("Applying Grad-CAM to validation samples...")

    # Pick random samples
    indices = np.random.choice(len(X_val), min(n_samples, len(X_val)), replace=False)
    
    class_names = {0: 'HEALTHY', 1: 'LEAFBLAST'}

    # Setup Plot
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 5 * n_samples))
    if n_samples == 1: axes = np.expand_dims(axes, axis=0)

    for row_idx, idx in enumerate(indices):
        img_array_sample = X_val[idx:idx+1]
        actual_label = y_val[idx]
        
        # Prediction
        pred_proba = model.predict(img_array_sample, verbose=0)[0][0]
        pred_class = 1 if pred_proba > 0.5 else 0
        
        # Generate Heatmap
        heatmap = enhanced_gradcam_heatmap(img_array_sample, model)
        
        # Resize heatmap to match image size (224x224)
        heatmap_resized = cv2.resize(heatmap, (TARGET_SIZE[0], TARGET_SIZE[1]))
        
        # Prepare Image
        # Assuming X_val is 0-1 float, convert to 0-255 uint8
        original_img = (img_array_sample[0] * 255).astype(np.uint8)
        
        # Create Overlay
        # Convert heatmap to RGB (Jet colormap)
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
        
        # Blend: 60% Original + 40% Heatmap
        superimposed = cv2.addWeighted(original_img, 0.6, heatmap_colored, 0.4, 0)

        # Display
        # Column 1: Original
        axes[row_idx, 0].imshow(original_img)
        axes[row_idx, 0].set_title(f'Actual: {class_names[actual_label]}', fontsize=12)
        axes[row_idx, 0].axis('off')

        # Column 2: Heatmap Only
        axes[row_idx, 1].imshow(heatmap_resized, cmap='jet')
        axes[row_idx, 1].set_title('Model Attention', fontsize=12)
        axes[row_idx, 1].axis('off')

        # Column 3: Overlay
        axes[row_idx, 2].imshow(superimposed)
        axes[row_idx, 2].set_title(f'Pred: {class_names[pred_class]} ({pred_proba:.2f})', fontsize=12)
        axes[row_idx, 2].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'gradcam_validation_final.png'), dpi=300, bbox_inches='tight')
    plt.show()

# Run it
apply_gradcam_validation(X_val, y_val, baseline_model, n_samples=6)

In [ ]:
def comparative_explainability_validation(baseline_model, plsr_model, xgb_model, X_val, y_val, n_samples=6):
    """Compare all three models on validation set"""
    print("="*60)
    print("COMPARATIVE EXPLAINABILITY - VALIDATION SET")
    print("="*60)
    
    # Select validation samples
    indices = np.random.choice(len(X_val), min(n_samples, len(X_val)), replace=False)
    
    fig, axes = plt.subplots(n_samples, 4, figsize=(18, 4 * n_samples))
    if n_samples == 1:
        axes = np.expand_dims(axes, axis=0)
    
    class_names = {0: 'HEALTHY', 1: 'LEAFBLAST'}
    
    for row_idx, idx in enumerate(indices):
        img_array = X_val[idx:idx+1]
        actual_label = y_val[idx]
        actual_class = class_names[actual_label]
        
        # Get predictions from ALL models
        baseline_pred = baseline_model.predict(img_array, verbose=0)[0][0]
        plsr_pred = get_plsr_prediction(baseline_model, plsr_model, plsr_scaler, img_array)
        xgb_pred = get_xgb_prediction(baseline_model, xgb_model, xgb_scaler, img_array)
        
        # Generate attention maps
        baseline_heatmap = enhanced_gradcam_heatmap(img_array, baseline_model)
        plsr_heatmap = generate_plsr_attention(baseline_model, plsr_model, img_array)
        xgb_heatmap = generate_xgb_attention(baseline_model, xgb_model, img_array)
        
        # Original image
        axes[row_idx, 0].imshow(img_array[0])
        axes[row_idx, 0].set_title(f'Actual: {actual_class}', fontsize=12, fontweight='bold')
        axes[row_idx, 0].axis('off')
        
        # Baseline VGG16
        plot_model_attention_validation(axes[row_idx, 1], img_array[0], baseline_heatmap, 
                                      baseline_pred, 'VGG16 Baseline', class_names)
        
        # VGG16 + PLSR  
        plot_model_attention_validation(axes[row_idx, 2], img_array[0], plsr_heatmap,
                                      plsr_pred, 'VGG16 + PLSR', class_names)
        
        # VGG16 + XGBoost
        plot_model_attention_validation(axes[row_idx, 3], img_array[0], xgb_heatmap,
                                      xgb_pred, 'VGG16 + XGBoost', class_names)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'comparative_explainability_validation.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

# Add these function definitions in 3.2 section

def generate_plsr_attention(baseline_model, plsr_model, img_array):
    """Create attention map showing what PLSR focuses on"""
    features = extract_features_correct(baseline_model, img_array)
    
    if len(features.shape) > 2:
        original_shape = features.shape[1:]  # e.g., (7, 7, 512)
        features_flat = features.reshape(features.shape[0], -1)
        
        # Get PLSR coefficients and project back to spatial dimensions
        plsr_coef = np.abs(plsr_model.coef_.flatten())
        
        # Handle shape mismatch by taking the first n features
        n_features = min(plsr_coef.shape[0], features_flat.shape[1])
        spatial_weights = plsr_coef[:n_features].reshape(original_shape[0], original_shape[1], -1)
        spatial_weights = spatial_weights.mean(axis=2)  # Average across channels
    else:
        # Fallback if features are already flat
        spatial_weights = np.ones((7, 7))
    
    # Resize to match original image
    attention_map = cv2.resize(spatial_weights, (224, 224))
    
    # Apply smoothing and normalization
    attention_map = cv2.GaussianBlur(attention_map, (15, 15), 3)
    if np.max(attention_map) > 0:
        attention_map = attention_map / np.max(attention_map)
    
    return attention_map

def generate_xgb_attention(baseline_model, xgb_model, img_array):
    """Create attention map showing what XGBoost focuses on"""
    if not XGB_AVAILABLE or xgb_model is None:
        return np.zeros((224, 224))
    
    features = extract_features_correct(baseline_model, img_array)
    
    if len(features.shape) > 2:
        original_shape = features.shape[1:]  # e.g., (7, 7, 512)
        features_flat = features.reshape(features.shape[0], -1)
        
        # Get XGBoost feature importances
        if hasattr(xgb_model, 'feature_importances_'):
            xgb_importance = xgb_model.feature_importances_
            n_features = min(xgb_importance.shape[0], features_flat.shape[1])
            spatial_weights = xgb_importance[:n_features].reshape(original_shape[0], original_shape[1], -1)
            spatial_weights = spatial_weights.mean(axis=2)
        else:
            # Fallback: use uniform weights
            spatial_weights = np.ones((original_shape[0], original_shape[1]))
    else:
        # Fallback if features are already flat
        spatial_weights = np.ones((7, 7))
    
    # Resize and process
    attention_map = cv2.resize(spatial_weights, (224, 224))
    attention_map = cv2.GaussianBlur(attention_map, (15, 15), 3)
    if np.max(attention_map) > 0:
        attention_map = attention_map / np.max(attention_map)
    
    return attention_map

# Add helper functions needed for 3.2
def get_plsr_prediction(baseline_model, plsr_model, scaler, img_array):
    """Get PLSR prediction for a single image"""
    features = extract_features_correct(baseline_model, img_array)
    if len(features.shape) > 2:
        features = features.reshape(features.shape[0], -1)
    features_scaled = scaler.transform(features)
    return plsr_model.predict(features_scaled)[0][0]

def get_xgb_prediction(baseline_model, xgb_model, scaler, img_array):
    """Get XGBoost prediction for a single image"""
    if not XGB_AVAILABLE or xgb_model is None:
        return 0.5
    features = extract_features_correct(baseline_model, img_array)
    if len(features.shape) > 2:
        features = features.reshape(features.shape[0], -1)
    features_scaled = scaler.transform(features)
    return xgb_model.predict_proba(features_scaled)[0][1]

def plot_model_attention_validation(ax, original_img, heatmap, prediction, model_name, class_names):
    """Plot individual model attention for validation set"""
    pred_class = 1 if prediction > 0.5 else 0
    pred_label = class_names[pred_class]
    confidence = prediction if pred_class == 1 else 1 - prediction
    
    original_uint8 = (original_img * 255).astype(np.uint8)
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed = cv2.addWeighted(original_uint8, 0.6, heatmap_colored, 0.4, 0)
    
    ax.imshow(superimposed)
    ax.set_title(f'{model_name}\nPred: {pred_label}\n({confidence:.3f})', fontsize=10)
    ax.axis('off')

# Call it in 3.2 after the original Grad-CAM
print("Running comparative explainability on validation set...")
comparative_explainability_validation(baseline_model, plsr_model, xgb_model, X_val, y_val)

## 4. Model Deployment Module

### 4.1 Optimal Model run with Testing set

In [ ]:
def prepare_test_data(ph_processed):
    """Prepare test data from the processed Philippines dataset"""
    print("Preparing test data from processed images...")

    X_test = []
    y_test = []
    test_info = []

    for img_data in ph_processed:
        # Use the already processed image from background removal
        processed_img = img_data['processed_image']
        label = auto_detect_label(img_data['original_path'])
        label_val = 1 if label == 'LEAFBLAST' else 0

        if processed_img is not None:
            X_test.append(processed_img)
            y_test.append(label_val)
            test_info.append({
                'original_path': img_data['original_path'],
                'annotated_name': create_annotation_format(
                    Path(img_data['original_path']).name,
                    label,
                    'LOCAL',
                    len(test_info) + 1
                ),
                'label': label,
                'label_numeric': label_val
            })

    X_test = np.array(X_test)
    y_test = np.array(y_test)

    print(f"Test data prepared: {X_test.shape[0]} images")
    return X_test, y_test, test_info

def evaluate_model_testing(model, model_type, X_test, y_test, feature_extractor=None, scaler=None):
    """Comprehensive model evaluation on test set"""
    start_time = time.time()

    if model_type == 'cnn':
        # CNN model (Baseline VGG16)
        y_pred_proba = model.predict(X_test, verbose=0)
        y_pred = (y_pred_proba > 0.5).astype(int).flatten()
        inference_time = time.time() - start_time

    elif model_type in ['plsr', 'xgboost']:
        # Feature-based models
        if feature_extractor is None or scaler is None:
            raise ValueError("Feature extractor and scaler required for feature-based models")

        # Extract features
        test_features = extract_features_correct(feature_extractor, X_test)
        if len(test_features.shape) > 2:
            test_features = test_features.reshape(test_features.shape[0], -1)

        test_features_scaled = scaler.transform(test_features)

        if model_type == 'plsr':
            y_pred_proba = model.predict(test_features_scaled)
            y_pred = (y_pred_proba > 0.5).astype(int).flatten()
        else:  # xgboost
            y_pred = model.predict(test_features_scaled)
            y_pred_proba = model.predict_proba(test_features_scaled)[:, 1]

        inference_time = time.time() - start_time

    # Calculate comprehensive metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Specificity (True Negative Rate)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    results = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'specificity': specificity,
        'inference_time': inference_time,
        'predictions': y_pred,
        'probabilities': y_pred_proba,
        'confusion_matrix': confusion_matrix(y_test, y_pred)
    }

    return results

def deploy_all_models_testing(baseline_model, plsr_model, xgb_model, test_df):
    """Deploy all models on test set and compare performance"""
    print("="*60)
    print("MODEL DEPLOYMENT - TEST SET EVALUATION")
    print("="*60)

    # Prepare test data
    X_test, y_test, test_info = prepare_test_data(ph_processed)

    test_results = {}

    # 1. Evaluate Baseline VGG16
    print("\n1. Testing Baseline VGG16 Model...")
    baseline_test_results = evaluate_model_testing(
        baseline_model, 'cnn', X_test, y_test
    )
    test_results['baseline'] = baseline_test_results

    print(f"  ✓ Accuracy: {baseline_test_results['accuracy']:.4f}")
    print(f"  ✓ Precision: {baseline_test_results['precision']:.4f}")
    print(f"  ✓ Recall: {baseline_test_results['recall']:.4f}")
    print(f"  ✓ F1-Score: {baseline_test_results['f1_score']:.4f}")
    print(f"  ✓ Specificity: {baseline_test_results['specificity']:.4f}")
    print(f"  ✓ Inference Time: {baseline_test_results['inference_time']:.2f}s")

    # 2. Evaluate PLSR Model
    print("\n2. Testing PLSR Model...")
    plsr_test_results = evaluate_model_testing(
        plsr_model, 'plsr', X_test, y_test,
        feature_extractor=baseline_model, scaler=plsr_scaler
    )
    test_results['plsr'] = plsr_test_results

    print(f"  ✓ Accuracy: {plsr_test_results['accuracy']:.4f}")
    print(f"  ✓ Precision: {plsr_test_results['precision']:.4f}")
    print(f"  ✓ Recall: {plsr_test_results['recall']:.4f}")
    print(f"  ✓ F1-Score: {plsr_test_results['f1_score']:.4f}")
    print(f"  ✓ Specificity: {plsr_test_results['specificity']:.4f}")
    print(f"  ✓ Inference Time: {plsr_test_results['inference_time']:.2f}s")

    # 3. Evaluate XGBoost Model
    if XGB_AVAILABLE and xgb_model is not None:
        print("\n3. Testing XGBoost Model...")
        xgb_test_results = evaluate_model_testing(
            xgb_model, 'xgboost', X_test, y_test,
            feature_extractor=baseline_model, scaler=xgb_scaler
        )
        test_results['xgboost'] = xgb_test_results

        print(f"  ✓ Accuracy: {xgb_test_results['accuracy']:.4f}")
        print(f"  ✓ Precision: {xgb_test_results['precision']:.4f}")
        print(f"  ✓ Recall: {xgb_test_results['recall']:.4f}")
        print(f"  ✓ F1-Score: {xgb_test_results['f1_score']:.4f}")
        print(f"  ✓ Specificity: {xgb_test_results['specificity']:.4f}")
        print(f"  ✓ Inference Time: {xgb_test_results['inference_time']:.2f}s")

    # Create comprehensive test results visualization
    plot_test_results_comprehensive(test_results, y_test)

    # Save test results as pickle
    test_results_path = os.path.join(OUTPUT_BASE, 'test_results.pkl')
    joblib.dump(test_results, test_results_path)
    print(f"✓ Test results saved as: {test_results_path}")

    # Save test info
    test_info_path = os.path.join(OUTPUT_BASE, 'test_info.pkl')
    joblib.dump(test_info, test_info_path)
    print(f"✓ Test info saved as: {test_info_path}")

    return test_results, X_test, y_test, test_info

def plot_test_results_comprehensive(test_results, y_test):
    """Create comprehensive visualization of test results - ALL MODELS IN ONE FIGURE"""
    models = list(test_results.keys())
    
    # Create a single comprehensive figure with dynamic layout
    n_models = len(models)
    n_cols = 2  # Metrics and inference time
    n_rows = 2 + (n_models + 1) // 2  # Adjust rows based on number of models
    
    fig = plt.figure(figsize=(20, 6 * n_rows))
    
    # Create grid specification for flexible layout
    gs = fig.add_gridspec(n_rows, 2)
    
    # Plot 1: Performance metrics comparison (top left)
    ax1 = fig.add_subplot(gs[0, 0])
    metrics = ['accuracy', 'precision', 'recall', 'f1_score']
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    
    x = np.arange(len(models))
    width = 0.2
    
    for i, metric in enumerate(metrics):
        values = [test_results[model][metric] for model in models]
        ax1.bar(x + (i-1.5)*width, values, width, label=metric_labels[i], alpha=0.8)
    
    ax1.set_xlabel('Models')
    ax1.set_ylabel('Score')
    ax1.set_title('Test Set Performance Metrics', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels([model.upper() for model in models])
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)
    
    # Plot 2: Inference time comparison (top right)
    ax2 = fig.add_subplot(gs[0, 1])
    inference_times = [test_results[model]['inference_time'] for model in models]
    bars = ax2.bar(models, inference_times, color='orange', alpha=0.7)
    ax2.set_xlabel('Models')
    ax2.set_ylabel('Seconds')
    ax2.set_title('Inference Time Comparison', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, time_val in zip(bars, inference_times):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{time_val:.2f}s', ha='center', va='bottom')
    
    # Plot 3+: Confusion matrices for ALL models
    for idx, model in enumerate(models):
        row = 1 + idx // 2  # Start from row 1, 2 models per row
        col = idx % 2
        
        ax = fig.add_subplot(gs[row, col])
        cm = test_results[model]['confusion_matrix']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                   xticklabels=['Healthy', 'Blast'],
                   yticklabels=['Healthy', 'Blast'])
        ax.set_title(f'{model.upper()} Confusion Matrix', fontsize=12, fontweight='bold')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    
    # Add model agreement analysis if available
    if 'model_agreement' in globals() or 'agreements' in globals():
        try:
            ax_agreement = fig.add_subplot(gs[-1, :])  # Bottom row, full width
            
            agreement_data = {
                'Full Agreement': len(agreements),
                'Disagreements': len(disagreements)
            }
            
            colors = ['green', 'red']
            bars = ax_agreement.bar(agreement_data.keys(), agreement_data.values(), 
                                   color=colors, alpha=0.7)
            
            ax_agreement.set_ylabel('Number of Samples')
            ax_agreement.set_title('Model Agreement Analysis on Test Set', 
                                 fontsize=14, fontweight='bold')
            ax_agreement.grid(True, alpha=0.3)
            
            # Add value labels
            for bar, value in zip(bars, agreement_data.values()):
                height = bar.get_height()
                ax_agreement.text(bar.get_x() + bar.get_width()/2., height + 5,
                                f'{value} samples', ha='center', va='bottom')
                
            # Add percentage labels
            total = sum(agreement_data.values())
            for i, (key, value) in enumerate(agreement_data.items()):
                percentage = (value / total) * 100
                ax_agreement.text(i, value/2, f'{percentage:.1f}%', 
                                ha='center', va='center', fontweight='bold', color='white')
                
        except NameError:
            pass  # Skip if agreement data not available
    
    plt.tight_layout()
    
    # Save the single comprehensive figure
    plt.savefig(os.path.join(OUTPUT_BASE, 'test_results_comprehensive_all_models.png'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# Run deployment testing
print("Starting model deployment on test set...")
test_results, X_test, y_test, test_info = deploy_all_models_testing(
    baseline_model, plsr_model, xgb_model, test_df
)
print("Model deployment testing completed successfully!")

### 4.2 Explainability Analysis

In [ ]:
def comparative_explainability_test_set(baseline_model, plsr_model, xgb_model, X_test, y_test, test_info, n_samples=8):
    """Comprehensive comparative explainability on test set"""
    print("="*60)
    print("COMPARATIVE EXPLAINABILITY - TEST SET")
    print("="*60)
    
    # Select diverse test samples (correct/incorrect predictions)
    test_predictions = baseline_model.predict(X_test, verbose=0).flatten()
    test_pred_classes = (test_predictions > 0.5).astype(int)
    
    correct_indices = np.where(test_pred_classes == y_test)[0]
    incorrect_indices = np.where(test_pred_classes != y_test)[0]
    
    n_each = min(n_samples // 2, len(correct_indices), len(incorrect_indices))
    correct_sample = np.random.choice(correct_indices, n_each, replace=False)
    incorrect_sample = np.random.choice(incorrect_indices, n_each, replace=False)
    selected_indices = np.concatenate([correct_sample, incorrect_sample])
    
    fig, axes = plt.subplots(len(selected_indices), 4, figsize=(18, 4 * len(selected_indices)))
    if len(selected_indices) == 1:
        axes = np.expand_dims(axes, axis=0)
    
    class_names = {0: 'HEALTHY', 1: 'LEAFBLAST'}
    
    for row_idx, idx in enumerate(selected_indices):
        img_array = X_test[idx:idx+1]
        actual_label = y_test[idx]
        actual_class = class_names[actual_label]
        
        # Get ALL model predictions
        baseline_pred = baseline_model.predict(img_array, verbose=0)[0][0]
        plsr_pred = get_plsr_prediction(baseline_model, plsr_model, plsr_scaler, img_array)
        xgb_pred = get_xgb_prediction(baseline_model, xgb_model, xgb_scaler, img_array)
        
        # Generate ALL attention maps
        baseline_heatmap = enhanced_gradcam_heatmap(img_array, baseline_model)
        plsr_heatmap = generate_plsr_attention(baseline_model, plsr_model, img_array)
        xgb_heatmap = generate_xgb_attention(baseline_model, xgb_model, img_array)
        
        # Check if baseline prediction is correct
        baseline_correct = (baseline_pred > 0.5) == actual_label
        result_color = 'green' if baseline_correct else 'red'
        result_text = 'correct' if baseline_correct else 'wrong'
        
        # Original image with correctness indicator
        axes[row_idx, 0].imshow(img_array[0])
        axes[row_idx, 0].set_title(f'Actual: {actual_class}\n{result_text}', 
                                 color=result_color, fontsize=12, fontweight='bold')
        axes[row_idx, 0].axis('off')
        
        # All three models
        plot_model_attention_test(axes[row_idx, 1], img_array[0], baseline_heatmap, 
                                baseline_pred, 'VGG16 Baseline', class_names)
        plot_model_attention_test(axes[row_idx, 2], img_array[0], plsr_heatmap,
                                plsr_pred, 'VGG16 + PLSR', class_names)
        plot_model_attention_test(axes[row_idx, 3], img_array[0], xgb_heatmap,
                                xgb_pred, 'VGG16 + XGBoost', class_names)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'comparative_explainability_test_set.png'),
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # Run model agreement analysis
    agreements, disagreements = analyze_model_agreement(baseline_model, plsr_model, xgb_model, X_test, y_test)

def plot_model_attention_test(ax, original_img, heatmap, prediction, model_name, class_names):
    """Plot individual model attention for test set with enhanced styling"""
    pred_class = 1 if prediction > 0.5 else 0
    pred_label = class_names[pred_class]
    confidence = prediction if pred_class == 1 else 1 - prediction
    
    original_uint8 = (original_img * 255).astype(np.uint8)
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed = cv2.addWeighted(original_uint8, 0.6, heatmap_colored, 0.4, 0)
    
    ax.imshow(superimposed)
    
    # Color code based on confidence
    confidence_color = 'green' if confidence > 0.7 else 'orange' if confidence > 0.5 else 'red'
    
    ax.set_title(f'{model_name}\n{pred_label}\nConf: {confidence:.3f}', 
                fontsize=10, color=confidence_color, fontweight='bold')
    ax.axis('off')

# Add the PLSR and XGBoost attention generation functions here
def generate_plsr_attention(baseline_model, plsr_model, img_array):
    """Create attention map showing what PLSR focuses on"""
    features = extract_features_correct(baseline_model, img_array)
    
    if len(features.shape) > 2:
        original_shape = features.shape[1:]  # e.g., (7, 7, 512)
        features_flat = features.reshape(features.shape[0], -1)
        
        # Get PLSR coefficients and project back to spatial dimensions
        plsr_coef = np.abs(plsr_model.coef_.flatten())
        
        # Handle shape mismatch by taking the first n features
        n_features = min(plsr_coef.shape[0], features_flat.shape[1])
        spatial_weights = plsr_coef[:n_features].reshape(original_shape[0], original_shape[1], -1)
        spatial_weights = spatial_weights.mean(axis=2)  # Average across channels
    else:
        # Fallback if features are already flat
        spatial_weights = np.ones((7, 7))
    
    # Resize to match original image
    attention_map = cv2.resize(spatial_weights, (224, 224))
    
    # Apply smoothing and normalization
    attention_map = cv2.GaussianBlur(attention_map, (15, 15), 3)
    if np.max(attention_map) > 0:
        attention_map = attention_map / np.max(attention_map)
    
    return attention_map

def generate_xgb_attention(baseline_model, xgb_model, img_array):
    """Create attention map showing what XGBoost focuses on"""
    if not XGB_AVAILABLE or xgb_model is None:
        return np.zeros((224, 224))
    
    features = extract_features_correct(baseline_model, img_array)
    
    if len(features.shape) > 2:
        original_shape = features.shape[1:]  # e.g., (7, 7, 512)
        features_flat = features.reshape(features.shape[0], -1)
        
        # Get XGBoost feature importances
        if hasattr(xgb_model, 'feature_importances_'):
            xgb_importance = xgb_model.feature_importances_
            n_features = min(xgb_importance.shape[0], features_flat.shape[1])
            spatial_weights = xgb_importance[:n_features].reshape(original_shape[0], original_shape[1], -1)
            spatial_weights = spatial_weights.mean(axis=2)
        else:
            # Fallback: use uniform weights
            spatial_weights = np.ones((original_shape[0], original_shape[1]))
    else:
        # Fallback if features are already flat
        spatial_weights = np.ones((7, 7))
    
    # Resize and process
    attention_map = cv2.resize(spatial_weights, (224, 224))
    attention_map = cv2.GaussianBlur(attention_map, (15, 15), 3)
    if np.max(attention_map) > 0:
        attention_map = attention_map / np.max(attention_map)
    
    return attention_map

# Add model agreement analysis
def analyze_model_agreement(baseline_model, plsr_model, xgb_model, X_test, y_test):
    """Analyze where models agree/disagree"""
    agreements = []
    disagreements = []
    
    # ADD THIS LINE: Define class_names inside the function
    class_names = {0: 'HEALTHY', 1: 'LEAFBLAST'}
    
    for i in range(len(X_test)):
        img_array = X_test[i:i+1]
        actual = y_test[i]
        
        # Get all predictions
        baseline_pred = baseline_model.predict(img_array, verbose=0)[0][0]
        plsr_pred = get_plsr_prediction(baseline_model, plsr_model, plsr_scaler, img_array)
        xgb_pred = get_xgb_prediction(baseline_model, xgb_model, xgb_scaler, img_array)
        
        pred_classes = [baseline_pred > 0.5, plsr_pred > 0.5, xgb_pred > 0.5]
        
        # Check agreement
        if len(set(pred_classes)) == 1:  # All models agree
            agreements.append(i)
        else:
            disagreements.append({
                'index': i, 'actual': actual, 'predictions': pred_classes,
                'confidence': [baseline_pred, plsr_pred, xgb_pred]
            })
    
    print(f"\nModel Agreement Analysis:")
    print(f"Total test samples: {len(X_test)}")
    print(f"Full agreement: {len(agreements)} ({len(agreements)/len(X_test)*100:.1f}%)")
    print(f"Disagreements: {len(disagreements)} ({len(disagreements)/len(X_test)*100:.1f}%)")
    
    # Show some disagreement cases
    if disagreements:
        print(f"\nShowing {min(3, len(disagreements))} disagreement cases:")
        for i, case in enumerate(disagreements[:3]):
            print(f"Case {i+1}: Actual={class_names[case['actual']]}, "
                  f"Preds=[VGG16:{case['predictions'][0]}, PLSR:{case['predictions'][1]}, XGB:{case['predictions'][2]}]")
    
    return agreements, disagreements

# Replace the current deployment explainability call with:
print("Running enhanced comparative explainability on test set...")
comparative_explainability_test_set(baseline_model, plsr_model, xgb_model, X_test, y_test, test_info)

## 5. Reporting Module

In [ ]:
def generate_comprehensive_report(validation_results, test_results, train_df, val_df, test_df):
    """Generate final comprehensive performance report"""
    print("="*60)
    print("COMPREHENSIVE PERFORMANCE REPORT")
    print("="*60)

    # Create results comparison dataframe
    report_data = []
    models = list(test_results.keys())

    for model in models:
        if model in validation_results and model in test_results:
            val_result = validation_results[model]
            test_result = test_results[model]

            report_data.append({
                'Model': model.upper(),
                'Val_Accuracy': val_result['accuracy'],
                'Test_Accuracy': test_result['accuracy'],
                'Test_Precision': test_result['precision'],
                'Test_Recall': test_result['recall'],
                'Test_F1_Score': test_result['f1_score'],
                'Test_Specificity': test_result['specificity'],  # Already included
                'Inference_Time_Seconds': test_result['inference_time'],
                'Performance_Gap': val_result['accuracy'] - test_result['accuracy']
            })

    report_df = pd.DataFrame(report_data)

    # Display the report table
    print("\nPERFORMANCE COMPARISON ACROSS MODELS:")
    print("="*50)
    print(report_df.round(4))

    # Dataset statistics
    print("\n" + "="*50)
    print("DATASET STATISTICS")
    print("="*50)

    dataset_stats = {
        'Dataset': ['Training (foreign)', 'Validation (foreign)', 'Testing (Philippines)'],
        'Total_Images': [len(train_df), len(val_df), len(test_df)],
        'LEAFBLAST_Count': [
            len(train_df[train_df['label'] == 'LEAFBLAST']),
            len(val_df[val_df['label'] == 'LEAFBLAST']),
            len(test_df[test_df['label'] == 'LEAFBLAST'])
        ],
        'HEALTHY_Count': [
            len(train_df[train_df['label'] == 'HEALTHY']),
            len(val_df[val_df['label'] == 'HEALTHY']),
            len(test_df[test_df['label'] == 'HEALTHY'])
        ],
        'LEAFBLAST_Percentage': [
            len(train_df[train_df['label'] == 'LEAFBLAST']) / len(train_df) * 100,
            len(val_df[val_df['label'] == 'LEAFBLAST']) / len(val_df) * 100,
            len(test_df[test_df['label'] == 'LEAFBLAST']) / len(test_df) * 100
        ]
    }

    stats_df = pd.DataFrame(dataset_stats)
    print(stats_df.round(2))

    # Model comparison analysis
    print("\n" + "="*50)
    print("MODEL COMPARISON ANALYSIS")
    print("="*50)

    best_test_accuracy = report_df['Test_Accuracy'].max()
    best_model = report_df.loc[report_df['Test_Accuracy'].idxmax(), 'Model']

    print(f"Best Performing Model: {best_model} (Test Accuracy: {best_test_accuracy:.4f})")

    for _, row in report_df.iterrows():
        model = row['Model']
        test_acc = row['Test_Accuracy']
        perf_gap = row['Performance_Gap']

        if model != best_model:
            diff = test_acc - best_test_accuracy
            if abs(diff) < 0.01:
                comparison = "COMPARABLE TO BEST"
            elif diff > -0.02:
                comparison = "SLIGHTLY WORSE"
            elif diff > -0.05:
                comparison = "WORSE"
            else:
                comparison = "MUCH WORSE"

            print(f"{model} vs {best_model}: {comparison} (Δ = {diff:+.4f})")

        # Performance gap analysis
        if perf_gap > 0.1:
            gap_analysis = "LARGE OVERFITTING"
        elif perf_gap > 0.05:
            gap_analysis = "MODERATE OVERFITTING"
        elif perf_gap > 0.02:
            gap_analysis = "SLIGHT OVERFITTING"
        elif perf_gap > -0.02:
            gap_analysis = "GOOD GENERALIZATION"
        else:
            gap_analysis = "UNDERFITTING"

        print(f"  {model} Generalization: {gap_analysis} (Val-Test gap: {perf_gap:+.4f})")

    # Create comprehensive visualization
    create_final_report_visualization(report_df, stats_df, validation_results, test_results)

    # Save detailed reports
    report_df.to_csv(os.path.join(OUTPUT_BASE, 'final_performance_report.csv'), index=False)
    stats_df.to_csv(os.path.join(OUTPUT_BASE, 'dataset_statistics.csv'), index=False)

    # Save model performance summary
    performance_summary = {
        'best_model': best_model,
        'best_accuracy': float(best_test_accuracy),
        'total_training_samples': len(train_df),
        'total_test_samples': len(test_df),
        'report_generated': time.strftime('%Y-%m-%d %H:%M:%S'),
        'models_evaluated': models
    }

    with open(os.path.join(OUTPUT_BASE, 'performance_summary.json'), 'w') as f:
        json.dump(performance_summary, f, indent=2)

    print(f"\nFinal reports saved to: {OUTPUT_BASE}")
    print("\n" + "="*50)
    print("RICE LEAF BLAST DETECTION SYSTEM - COMPLETED SUCCESSFULLY!")
    print("="*50)

    return report_df, stats_df

def create_final_report_visualization(report_df, stats_df, validation_results, test_results):
    """Create comprehensive final report visualization"""
    fig = plt.figure(figsize=(20, 16))

    # Overall layout
    gs = fig.add_gridspec(3, 3)

    # Plot 1: Performance metrics comparison (test set) - UPDATED WITH SPECIFICITY
    ax1 = fig.add_subplot(gs[0, 0])
    metrics = ['Test_Accuracy', 'Test_Precision', 'Test_Recall', 'Test_F1_Score', 'Test_Specificity']  # Added Specificity
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']  # Added Specificity

    x = np.arange(len(report_df))
    width = 0.15  # Reduced width to accommodate 5 metrics instead of 4

    for i, metric in enumerate(metrics):
        values = report_df[metric].values
        ax1.bar(x + (i-2)*width, values, width, label=metric_labels[i], alpha=0.8)  # Adjusted positioning

    ax1.set_xlabel('Models')
    ax1.set_ylabel('Score')
    ax1.set_title('Test Set Performance Metrics', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(report_df['Model'].values)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)

    # Plot 2: Validation vs Test Accuracy
    ax2 = fig.add_subplot(gs[0, 1])
    x_pos = np.arange(len(report_df))
    width = 0.35

    ax2.bar(x_pos - width/2, report_df['Val_Accuracy'], width, label='Validation', alpha=0.7, color='blue')
    ax2.bar(x_pos + width/2, report_df['Test_Accuracy'], width, label='Test', alpha=0.7, color='red')

    ax2.set_xlabel('Models')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Validation vs Test Accuracy', fontsize=14, fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(report_df['Model'].values)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1)

    # Plot 3: Inference Time Comparison
    ax3 = fig.add_subplot(gs[0, 2])
    bars = ax3.bar(report_df['Model'], report_df['Inference_Time_Seconds'],
                  alpha=0.7, color='green')
    ax3.set_xlabel('Models')
    ax3.set_ylabel('Seconds')
    ax3.set_title('Inference Time Comparison', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)

    # Add value labels
    for bar, time_val in zip(bars, report_df['Inference_Time_Seconds']):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{time_val:.2f}s', ha='center', va='bottom')

    # Plot 4: Dataset Distribution
    ax4 = fig.add_subplot(gs[1, 0])
    datasets = stats_df['Dataset']
    healthy_counts = stats_df['HEALTHY_Count']
    blast_counts = stats_df['LEAFBLAST_Count']

    x = np.arange(len(datasets))
    width = 0.35

    ax4.bar(x - width/2, healthy_counts, width, label='Healthy', color='green', alpha=0.7)
    ax4.bar(x + width/2, blast_counts, width, label='Leaf Blast', color='red', alpha=0.7)

    ax4.set_xlabel('Dataset')
    ax4.set_ylabel('Number of Images')
    ax4.set_title('Dataset Class Distribution', fontsize=14, fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels([d.split(' ')[0] for d in datasets])  # Shorten labels
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    # Plot 5: Performance Gap Analysis
    ax5 = fig.add_subplot(gs[1, 1])
    colors = ['red' if gap > 0.05 else 'orange' if gap > 0.02 else 'green' for gap in report_df['Performance_Gap']]
    bars = ax5.bar(report_df['Model'], report_df['Performance_Gap'], color=colors, alpha=0.7)
    ax5.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax5.set_xlabel('Models')
    ax5.set_ylabel('Accuracy Difference (Val - Test)')
    ax5.set_title('Generalization Performance Gap', fontsize=14, fontweight='bold')
    ax5.grid(True, alpha=0.3)

    # Add value labels
    for bar, gap in zip(bars, report_df['Performance_Gap']):
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02),
                f'{gap:+.3f}', ha='center', va='bottom' if height >= 0 else 'top')

    # Plot 6: Detailed Metrics for Best Model - UPDATED WITH SPECIFICITY
    ax6 = fig.add_subplot(gs[1, 2])
    best_model = report_df.loc[report_df['Test_Accuracy'].idxmax(), 'Model'].lower()

    if best_model in test_results:
        metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']  # Added Specificity
        metrics_values = [
            test_results[best_model]['accuracy'],
            test_results[best_model]['precision'],
            test_results[best_model]['recall'],
            test_results[best_model]['f1_score'],
            test_results[best_model]['specificity']  # Added Specificity
        ]

        colors = ['blue', 'green', 'orange', 'red', 'purple']  # Added purple for Specificity
        bars = ax6.bar(metrics_names, metrics_values, color=colors, alpha=0.7)

        ax6.set_ylabel('Score')
        ax6.set_title(f'Best Model ({best_model.upper()}) Detailed Metrics', fontsize=14, fontweight='bold')
        ax6.set_ylim(0, 1)
        ax6.grid(True, alpha=0.3)

        # Add value labels
        for bar, value in zip(bars, metrics_values):
            height = bar.get_height()
            ax6.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{value:.3f}', ha='center', va='bottom')

    # Plot 7: Training History (if available)
    ax7 = fig.add_subplot(gs[2, :])
    if 'baseline_history' in globals():
        history = baseline_history.history
        epochs = range(1, len(history['accuracy']) + 1)

        ax7.plot(epochs, history['accuracy'], 'b-', label='Training Accuracy', linewidth=2)
        ax7.plot(epochs, history['val_accuracy'], 'r-', label='Validation Accuracy', linewidth=2)
        ax7.set_xlabel('Epochs')
        ax7.set_ylabel('Accuracy')
        ax7.set_title('Training History - Baseline VGG16', fontsize=14, fontweight='bold')
        ax7.legend()
        ax7.grid(True, alpha=0.3)
    else:
        ax7.text(0.5, 0.5, 'Training History Not Available',
                ha='center', va='center', transform=ax7.transAxes, fontsize=12)
        ax7.set_title('Training History', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, 'final_comprehensive_report.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

# Generate final comprehensive report
print("Generating comprehensive performance report...")
final_report_df, final_stats_df = generate_comprehensive_report(
    validation_results, test_results, train_df, val_df, test_df
)